# Heretic Qwen — Colab

This notebook uses the same checked-in runtime as Kaggle: **Qwen3.5 9B uncensored Q6_K, up to 32,768 context tokens by default, batch 128 and q8_0 KV**, plus image attachments, **Qwen-Image 2.1 Images mode**, live operational logs, chat/image switching and website shutdown controls. Thinking, saved chat history, automatic compaction, document reading and Answer now remain in your existing Cloudflare website.

1. Select a **GPU runtime with enough memory**. This smaller model targets a single 16GB T4. Model-file memory is checked before downloads. Context is tried at 32,768, then 24,576, 16,384 and 8,192 tokens, keeping the same 9B Q6_K model entirely on GPU. The website displays the context that actually loaded; available hardware is not guaranteed. Kaggle keeps its existing 27B profile.
2. In **Secrets** (the key icon), add **QWEN_RELAY_KEY** with your existing relay key and enable notebook access. Otherwise the server cell asks in a hidden input. Never put it in notebook source or share it.
3. Choose **Runtime → Run all** and keep the notebook connected. Return to the website for live startup logs, chat and Images mode. The notebook unloads chat before handing the GPU to the image model and restores chat when you switch back.

Model/runtime/image files already present in this Colab VM are reused. A fresh VM needs cloud-side downloads again and cannot mount your private Kaggle dataset. This notebook creates no paid cache and uses no external AI provider.

Colab determines hardware availability and runtime limits. Read [Colab's FAQ](https://research.google.com/colaboratory/faq.html) for service restrictions; a selected software profile cannot guarantee hardware or uptime. Stop the server cell and disconnect the Colab runtime when finished; stopping a reply alone does not disconnect its GPU runtime.


In [ ]:
# Colab uses the smaller Qwen 9B model. The notebook measures context fit on your GPU.
COLAB_CONFIG = {
    "RELAY": '',  # Your own deployment; asked interactively if blank.
    "CONTEXT": 32768,
    "MODEL_DIR": "/content/ollama-models",  # Reused by reruns in this VM.
}


In [ ]:
#@title Start Qwen chat, vision and Images with the same runtime as Kaggle
COLAB_RUNTIME_SOURCE = '"""Bounded, best-effort operational logs shared by Kaggle and Colab notebooks.\n\nOnly explicit log() calls are uploaded. Never wrap arbitrary stdout, model output,\nprompts, subprocess logs or environment variables. Logging never waits for HTTP.\n"""\n\nimport collections as _nl_collections\nimport datetime as _nl_datetime\nimport json as _nl_json\nimport re as _nl_re\nimport threading as _nl_threading\nimport urllib.error as _nl_error\nimport urllib.request as _nl_request\nimport uuid as _nl_uuid\n\n\ndef notebook_log_redact(message, secrets=()):\n    text = str(message)\n    for secret in sorted((str(value) for value in secrets if value), key=len, reverse=True):\n        text = text.replace(secret, "[redacted]")\n    text = _nl_re.sub(r"https?://[a-z0-9-]+\\.trycloudflare\\.com(?:/[^\\s]*)?", "[tunnel address hidden]", text, flags=_nl_re.I)\n    text = _nl_re.sub(r"\\bKGAT_[A-Za-z0-9_-]+", "[redacted]", text)\n    text = _nl_re.sub(r"\\bBearer\\s+[^\\s,;\\"\']+", "Bearer [redacted]", text, flags=_nl_re.I)\n    text = _nl_re.sub(r"(?i)([\\"\']?(?:x-relay-key|x-launch-token|relay_key|launch_token|api[_-]?key|api[_-]?token|authorization|access_token)[\\"\']?\\s*[:=]\\s*)[\\"\']?[^\\s,;\\"\']+", r"\\1[redacted]", text)\n    return _nl_re.sub(r"[\\x00-\\x08\\x0b-\\x1f\\x7f]", " ", text).encode("utf-8", "replace").decode("utf-8")\n\n\nclass NotebookLogReporter:\n    """Upload up to forty lines every five seconds without blocking the model server.\n\n    Failed batches retain the same sequence number, making retries idempotent.\n    During an outage only the newest 200 queued lines are retained, plus one\n    in-flight batch. Authentication/stale-run rejection stops the old reporter.\n    """\n\n    def __init__(self, relay, host, headers=None, secrets=(), run_id=None, interval=5, timeout=5):\n        if host not in ("kaggle", "colab"):\n            raise ValueError("unsupported notebook host")\n        if not str(relay).startswith("https://"):\n            raise ValueError("notebook logging needs an HTTPS relay")\n        self.relay = str(relay).rstrip("/")\n        self.host = host\n        self.headers = dict(headers or {})\n        self.secrets = tuple(secrets) + tuple(self.headers.values())\n        self.run_id = run_id or _nl_uuid.uuid4().hex\n        self.started_at = _nl_datetime.datetime.now(_nl_datetime.timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")\n        self.interval = max(1, float(interval))\n        self.timeout = min(10, max(1, float(timeout)))\n        self._queue = _nl_collections.deque(maxlen=200)\n        self._lock = _nl_threading.Lock()\n        self._wake = _nl_threading.Event()\n        self._pending = None\n        self._seq = 0\n        self._status = "RUNNING"\n        self._finishing = False\n        self._stopped = False\n        self._finish_failures = 0\n        self._thread = _nl_threading.Thread(target=self._run, name="notebook-operational-log", daemon=True)\n        self._thread.start()\n\n    def log(self, message):\n        """Queue an explicit operational message. No network or file I/O occurs here."""\n        clean = notebook_log_redact(message, self.secrets)\n        clock = _nl_datetime.datetime.now(_nl_datetime.timezone.utc).strftime("%H:%M:%S")\n        with self._lock:\n            if self._stopped or self._finishing:\n                return\n            for line in clean.splitlines()[:200]:\n                if line.strip():\n                    # The Worker counts JS UTF-16 units, including surrogate pairs.\n                    bounded = (clock + " " + line).encode("utf-16-le")[:4000].decode("utf-16-le", "ignore")\n                    self._queue.append(bounded)\n            full = len(self._queue) >= 40\n        if full:\n            self._wake.set()\n\n    def finish(self, status="COMPLETE", wait=0):\n        """Request a final flush, optionally waiting a bounded number of seconds.\n\n        The default is nonblocking. Notebook shutdown may pass wait=6 to give a\n        final batch one HTTP attempt before its Python process exits.\n        """\n        if status not in ("COMPLETE", "ERROR"):\n            raise ValueError("invalid notebook final status")\n        with self._lock:\n            if self._stopped or self._finishing:\n                return\n            self._status = status\n            self._finishing = True\n            clock = _nl_datetime.datetime.now(_nl_datetime.timezone.utc).strftime("%H:%M:%S")\n            self._queue.append(clock + (" Notebook stopped with an error." if status == "ERROR" else " Notebook session ended."))\n        self._wake.set()\n        if wait and self._thread is not _nl_threading.current_thread():\n            self._thread.join(timeout=min(10, max(0, float(wait))))\n\n    def _flush_once(self):\n        with self._lock:\n            if self._stopped:\n                return\n            if self._pending is None and self._queue:\n                lines = []\n                encoded_size = 0\n                while self._queue and len(lines) < 40:\n                    size = len(_nl_json.dumps(self._queue[0], ensure_ascii=False).encode("utf-8")) + 1\n                    if lines and encoded_size + size > 75 * 1024:\n                        break\n                    lines.append(self._queue.popleft())\n                    encoded_size += size\n                status = self._status if self._finishing and not self._queue else "RUNNING"\n                self._pending = {"host": self.host, "run_id": self.run_id, "started_at": self.started_at, "seq": self._seq, "lines": lines, "status": status}\n            payload = self._pending\n        if payload is None:\n            return\n        request = _nl_request.Request(self.relay + "/_relay/log", data=_nl_json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), method="POST", headers={"content-type": "application/json", "user-agent": "heretic-notebook-log/1", **self.headers})\n        try:\n            with _nl_request.urlopen(request, timeout=self.timeout) as response:\n                # No server response text is logged; errors may contain credentials.\n                if response.status != 200:\n                    raise OSError("log upload rejected")\n        except _nl_error.HTTPError as error:\n            if error.code in (401, 403, 409):\n                with self._lock:\n                    self._stopped = True\n                    self._queue.clear()\n                    self._pending = None\n            raise\n        with self._lock:\n            self._pending = None\n            self._seq += 1\n            self._finish_failures = 0\n            if self._finishing and not self._queue:\n                self._stopped = True\n            elif len(self._queue) >= 40 or self._finishing:\n                self._wake.set()\n\n    def _run(self):\n        while True:\n            self._wake.wait(self.interval)\n            self._wake.clear()\n            with self._lock:\n                if self._stopped:\n                    return\n            try:\n                self._flush_once()\n            except Exception:\n                # Logging is best effort. A network error never interrupts inference.\n                self._wake.clear()\n                with self._lock:\n                    if self._finishing:\n                        self._finish_failures += 1\n                        if self._finish_failures >= 3:\n                            self._stopped = True\n\n\n"""Read an immutable Kaggle cache; stage writable Ollama/ComfyUI stores locally.\n\nLarge payload hashes are checked by the cache builder, not on every startup.\nDiscovery returns {"root": absolute_path, "manifest": parsed_cache_json}, or None.\nModel, runtime and image staging fail independently with CacheError; callers retain\ntheir normal download/install fallback. Nothing writes to the mounted cache.\n"""\n\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport stat\nimport tarfile\nimport tempfile\n\n\nclass CacheError(ValueError):\n    pass\n\n\ndef _cache_relative(value):\n    if not isinstance(value, str) or not value or "\\\\" in value:\n        raise CacheError("cache path must be a relative POSIX path")\n    if value.startswith("/") or re.match(r"^[A-Za-z]:", value):\n        raise CacheError("absolute cache path")\n    if any(part in ("", ".", "..") for part in value.split("/")):\n        raise CacheError("unsafe cache path")\n    return value\n\n\ndef _cache_inside(root, relative):\n    root = os.path.realpath(root)\n    path = os.path.realpath(os.path.join(root, _cache_relative(relative)))\n    try:\n        contained = os.path.commonpath((root, path)) == root\n    except ValueError:\n        contained = False\n    if not contained:\n        raise CacheError("cache path escapes its root")\n    return path\n\n\ndef _cache_hash(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as source:\n        for block in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _cache_file(cache, entry, *, hash_small=False):\n    if not isinstance(entry, dict):\n        raise CacheError("invalid cache file descriptor")\n    size, digest = entry.get("size"), entry.get("sha256")\n    if type(size) is not int or size < 0 or not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):\n        raise CacheError("invalid cache file size or digest")\n    path = _cache_inside(cache["root"], entry.get("path"))\n    if not os.path.isfile(path) or os.path.getsize(path) != size:\n        raise CacheError("cache file missing or truncated: " + entry["path"])\n    if hash_small:\n        if size > 1024 * 1024:\n            raise CacheError("cache metadata exceeds 1 MiB")\n        if _cache_hash(path) != digest:\n            raise CacheError("cache metadata digest mismatch: " + entry["path"])\n    return path\n\n\ndef _cache_model(cache):\n    model = cache["manifest"].get("model")\n    if not isinstance(model, dict):\n        raise CacheError("model cache metadata missing")\n    if not isinstance(model.get("name"), str) or not re.fullmatch(r"[A-Za-z0-9_.-]+", model["name"]):\n        raise CacheError("invalid model name")\n    if not all(isinstance(model.get(key), str) and model[key] for key in ("repo", "revision", "quant")):\n        raise CacheError("model identity is incomplete")\n    files = model.get("files")\n    if not isinstance(files, list) or len(files) != 3 or any(not isinstance(item, dict) for item in files):\n        raise CacheError("model cache needs model, projector and config files")\n    by_role = {item.get("role"): item for item in files}\n    if set(by_role) != {"model", "projector", "config"}:\n        raise CacheError("invalid model cache roles")\n    paths = {role: _cache_file(cache, item, hash_small=role == "config") for role, item in by_role.items()}\n    manifest_path = _cache_inside(cache["root"], model.get("manifest"))\n    if not os.path.isfile(manifest_path) or os.path.getsize(manifest_path) > 1024 * 1024:\n        raise CacheError("model manifest missing or too large")\n    if _cache_hash(manifest_path) != model.get("manifest_sha256"):\n        raise CacheError("model manifest digest mismatch")\n    with open(manifest_path, encoding="utf-8") as source:\n        manifest = json.load(source)\n    if not isinstance(manifest, dict):\n        raise CacheError("invalid Ollama manifest")\n    expected_types = {\n        "model": "application/vnd.ollama.image.model",\n        "projector": "application/vnd.ollama.image.projector",\n    }\n    layers = manifest.get("layers")\n    if manifest.get("schemaVersion") != 2 or not isinstance(layers, list) or len(layers) != 2:\n        raise CacheError("unsupported Ollama manifest")\n    for role, media_type in expected_types.items():\n        entry = by_role[role]\n        matches = [layer for layer in layers if isinstance(layer, dict) and layer.get("mediaType") == media_type]\n        if len(matches) != 1 or matches[0].get("digest") != "sha256:" + entry["sha256"] or matches[0].get("size") != entry["size"]:\n            raise CacheError("Ollama manifest layer does not match cache")\n    config = manifest.get("config", {})\n    if not isinstance(config, dict) or config.get("digest") != "sha256:" + by_role["config"]["sha256"] or config.get("size") != by_role["config"]["size"]:\n        raise CacheError("Ollama manifest config does not match cache")\n    return model, by_role, paths, manifest_path\n\n\ndef validate_cache(root, *, repo, quant, model_name, cache_id=None):\n    root = os.path.realpath(root)\n    metadata = _cache_inside(root, "cache.json")\n    if os.path.getsize(metadata) > 1024 * 1024:\n        raise CacheError("cache metadata exceeds 1 MiB")\n    with open(metadata, encoding="utf-8") as source:\n        manifest = json.load(source)\n    if not isinstance(manifest, dict) or type(manifest.get("schema")) is not int or manifest["schema"] != 1:\n        raise CacheError("unsupported cache schema")\n    if not isinstance(manifest.get("cache_id"), str) or not manifest["cache_id"]:\n        raise CacheError("cache identity missing")\n    if cache_id is not None and manifest["cache_id"] != cache_id:\n        raise CacheError("cache identity mismatch")\n    cache = {"root": root, "manifest": manifest}\n    model, _, _, _ = _cache_model(cache)\n    if (model["repo"], model["quant"], model["name"]) != (repo, quant, model_name):\n        raise CacheError("cached model identity mismatch")\n    return cache\n\n\ndef discover_cache(input_root="/kaggle/input", *, repo, quant, model_name, cache_id=None, max_depth=3):\n    """Find a matching cache.json no more than three directory levels below input_root."""\n    if type(max_depth) is not int or not 0 <= max_depth <= 3:\n        raise ValueError("cache search depth must be between zero and three")\n    input_root = os.path.realpath(input_root)\n    for directory, children, files in os.walk(input_root, followlinks=False):\n        relative = os.path.relpath(directory, input_root)\n        depth = 0 if relative == "." else len(relative.split(os.sep))\n        children[:] = sorted(children) if depth < max_depth else []\n        if "cache.json" not in files:\n            continue\n        try:\n            return validate_cache(directory, repo=repo, quant=quant, model_name=model_name, cache_id=cache_id)\n        except (CacheError, OSError, ValueError, TypeError):\n            continue\n    return None\n\n\ndef _cache_copy_atomic(source, destination):\n    os.makedirs(os.path.dirname(destination), exist_ok=True)\n    handle, temporary = tempfile.mkstemp(prefix=".cache-", dir=os.path.dirname(destination))\n    os.close(handle)\n    try:\n        shutil.copyfile(source, temporary)\n        os.replace(temporary, destination)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _cache_output_root(cache, path):\n    path, source = os.path.realpath(path), os.path.realpath(cache["root"])\n    try:\n        common = os.path.commonpath((source, path))\n        inside_source = common == source or common == path\n    except ValueError:\n        inside_source = False\n    if inside_source:\n        raise CacheError("writable staging must be outside the immutable cache")\n    return path\n\n\ndef _cache_destination(root, relative):\n    relative = _cache_relative(relative)\n    parent, name = relative.rsplit("/", 1) if "/" in relative else (None, relative)\n    directory = _cache_inside(root, parent) if parent else root\n    return os.path.join(directory, name)\n\n\ndef _cache_link(source, destination, created):\n    os.makedirs(os.path.dirname(destination), exist_ok=True)\n    if os.path.lexists(destination):\n        if os.path.islink(destination) and os.path.realpath(destination) == os.path.realpath(source):\n            return\n        raise CacheError("cache staging would replace an existing file: " + destination)\n    os.symlink(source, destination)\n    created.append(destination)\n\n\ndef stage_cached_model(cache, writable_models):\n    """Return the installed manifest path; only immutable large blobs are symlinked."""\n    model, entries, sources, manifest = _cache_model(cache)\n    writable_models = _cache_output_root(cache, writable_models)\n    created = []\n    try:\n        for role in ("model", "projector"):\n            target = _cache_destination(writable_models, "blobs/sha256-" + entries[role]["sha256"])\n            _cache_link(sources[role], target, created)\n        _cache_copy_atomic(sources["config"], _cache_destination(writable_models, "blobs/sha256-" + entries["config"]["sha256"]))\n        target = _cache_destination(writable_models, "manifests/registry.ollama.ai/library/" + model["name"] + "/latest")\n        _cache_copy_atomic(manifest, target)\n        return target\n    except Exception:\n        for path in reversed(created):\n            os.unlink(path)\n        raise\n\n\ndef install_cached_runtime(cache, writable_runtime, *, ollama_version=None, cloudflared_version=None):\n    """Extract the cached runtime locally and return absolute {ollama, cloudflared} paths."""\n    runtime = cache["manifest"].get("runtime", {})\n    if not isinstance(runtime, dict) or runtime.get("platform") != "linux-amd64":\n        raise CacheError("unsupported cached runtime platform")\n    components = {}\n    for name, expected_version in (("ollama", ollama_version), ("cloudflared", cloudflared_version)):\n        entry = runtime.get(name)\n        if not isinstance(entry, dict) or not isinstance(entry.get("version"), str) or not entry["version"]:\n            raise CacheError("cached runtime version missing")\n        if expected_version is not None and entry["version"] != expected_version:\n            raise CacheError("cached runtime version mismatch")\n        components[name] = _cache_file(cache, entry)\n    executable = _cache_relative(runtime["ollama"].get("executable"))\n    writable_runtime = _cache_output_root(cache, writable_runtime)\n    os.makedirs(writable_runtime, exist_ok=True)\n    staged = tempfile.mkdtemp(prefix="cached-runtime-", dir=os.path.abspath(writable_runtime))\n    try:\n        with tarfile.open(components["ollama"], "r:") as archive:\n            if not hasattr(tarfile, "data_filter"):\n                raise CacheError("safe cached-runtime extraction requires tarfile.data_filter")\n            archive.extractall(staged, filter="data")\n        ollama = _cache_inside(staged, executable)\n        if not os.path.isfile(ollama):\n            raise CacheError("cached Ollama executable missing")\n        cloudflared = os.path.join(staged, "cloudflared")\n        _cache_copy_atomic(components["cloudflared"], cloudflared)\n        for path in (ollama, cloudflared):\n            os.chmod(path, os.stat(path).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)\n        return {"ollama": ollama, "cloudflared": cloudflared}\n    except Exception:\n        shutil.rmtree(staged)\n        raise\n\n\ndef validate_image_cache(cache, *, repo=None, revision=None):\n    """Validate images separately, so an unusable image cache cannot reject chat."""\n    images = cache["manifest"].get("images")\n    if not isinstance(images, dict) or not all(isinstance(images.get(k), str) and images[k] for k in ("repo", "revision")):\n        raise CacheError("image cache identity missing")\n    if (repo is not None and images["repo"] != repo) or (revision is not None and images["revision"] != revision):\n        raise CacheError("image cache identity mismatch")\n    files = images.get("files")\n    if not isinstance(files, list) or not files:\n        raise CacheError("image cache files missing")\n    targets = set()\n    for entry in files:\n        _cache_file(cache, entry)\n        source_path = _cache_relative(entry.get("source_path"))\n        folder = entry.get("folder")\n        if not isinstance(folder, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", folder):\n            raise CacheError("invalid ComfyUI model folder")\n        target = (folder, source_path.rsplit("/", 1)[-1])\n        if target in targets:\n            raise CacheError("duplicate image cache destination")\n        targets.add(target)\n    return images\n\n\ndef stage_cached_images(cache, comfy_models, *, repo=None, revision=None):\n    """Return {HuggingFace_source_path: local_path} for staged ComfyUI model files."""\n    images = validate_image_cache(cache, repo=repo, revision=revision)\n    comfy_models = _cache_output_root(cache, comfy_models)\n    result, created = {}, []\n    try:\n        for entry in images["files"]:\n            target = _cache_destination(comfy_models, entry["folder"] + "/" + entry["source_path"].rsplit("/", 1)[-1])\n            _cache_link(_cache_file(cache, entry), target, created)\n            result[entry["source_path"]] = target\n        return result\n    except Exception:\n        for path in reversed(created):\n            os.unlink(path)\n        raise\n\n\n"""Loopback Ollama proxy that keeps an NDJSON response alive during prefill.\n\nThis file is also concatenated before kaggle-server.py in a single namespace.\nKeep imports and implementation names private to avoid that script\'s globals.\n"""\n\nimport http.client as _stream_http\nimport http.server as _stream_server\nimport json as _stream_json\nimport math as _stream_math\nimport queue as _stream_queue\nimport select as _stream_select\nimport socket as _stream_socket\nimport threading as _stream_threading\nimport time as _stream_time\nimport urllib.parse as _stream_url\n\n\n_STREAM_HOP_HEADERS = frozenset((\n    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",\n    "te", "trailer", "transfer-encoding", "upgrade",\n))\n\n\ndef _stream_error_bytes(message, code=None):\n    error = {"error": message}\n    if code is not None:\n        error["code"] = code\n    return (_stream_json.dumps(error, ensure_ascii=False) + "\\n").encode("utf-8")\n\n\ndef _stream_terminal_record(record):\n    if not record.strip():\n        return None\n    try:\n        value = _stream_json.loads(record)\n    except (ValueError, UnicodeError):\n        return False\n    return isinstance(value, dict) and (value.get("done") is True or "error" in value)\n\n\ndef _stream_forward_headers(headers, request=False):\n    omitted = set(_STREAM_HOP_HEADERS)\n    for value in headers.get_all("Connection", []):\n        omitted.update(name.strip().lower() for name in value.split(","))\n    if request:\n        omitted.update(("host", "content-length", "expect"))\n    return [(name, value) for name, value in headers.items() if name.lower() not in omitted]\n\n\nclass _StreamRequest:\n    def __init__(self):\n        self.cancelled = _stream_threading.Event()\n        self.finished = _stream_threading.Event()\n        self.events = _stream_queue.Queue(maxsize=8)\n        self.lock = _stream_threading.Lock()\n        self.connection = None\n        self.socket = None\n        self.thread = None\n\n    def put(self, kind, value=None):\n        while not self.cancelled.is_set():\n            try:\n                self.events.put((kind, value), timeout=0.1)\n                return True\n            except _stream_queue.Full:\n                pass\n        return False\n\n    def set_connection(self, connection):\n        with self.lock:\n            self.connection = connection\n        if self.cancelled.is_set():\n            connection.close()\n            return False\n        return True\n\n    def set_socket(self, sock):\n        # Retain this even when HTTPConnection detaches a Connection: close\n        # response. Its HTTPResponse may still be blocked reading that socket.\n        with self.lock:\n            self.socket = sock\n        if self.cancelled.is_set():\n            self.cancel()\n            return False\n        return True\n\n    def cancel(self):\n        self.cancelled.set()\n        with self.lock:\n            sock, connection = self.socket, self.connection\n        if sock is not None:\n            try:\n                sock.shutdown(_stream_socket.SHUT_RDWR)\n            except OSError:\n                pass\n        if connection is not None:\n            # shutdown first: close alone does not interrupt getresponse/read1\n            # reliably when another thread owns the socket\'s buffered reader.\n            try:\n                connection.close()\n            except OSError:\n                pass\n\n\ndef _stream_upstream_error(body, status):\n    try:\n        parsed = _stream_json.loads(body)\n        if isinstance(parsed, dict) and "error" in parsed:\n            return body + (b"" if body.endswith(b"\\n") else b"\\n")\n        message = parsed if isinstance(parsed, str) else _stream_json.dumps(parsed, ensure_ascii=False)\n    except (ValueError, UnicodeError):\n        message = body.decode("utf-8", "replace").strip()\n    detail = (": " + message[:4096]) if message else ""\n    return _stream_error_bytes("Ollama returned HTTP %s%s" % (status, detail))\n\n\ndef _stream_run_upstream(server, state, method, path, headers, body, streaming):\n    response = None\n    connection = _stream_http.HTTPConnection(server.upstream_host, server.upstream_port, timeout=10)\n    try:\n        if not state.set_connection(connection):\n            return\n        connection.connect()\n        if not state.set_socket(connection.sock):\n            return\n        # Long prompt evaluation has no read deadline; the downstream handler\n        # remains responsive and cancels this socket when the downstream fails.\n        connection.sock.settimeout(None)\n        if state.cancelled.is_set():\n            return\n        connection.request(method, path, body=body, headers=headers)\n        response = connection.getresponse()\n        if state.cancelled.is_set():\n            return\n        if streaming and not 200 <= response.status < 300:\n            error_body = bytearray()\n            while not state.cancelled.is_set():\n                chunk = response.read1(min(65536, server.max_request_bytes + 1 - len(error_body)))\n                if not chunk:\n                    break\n                error_body.extend(chunk)\n                if len(error_body) > server.max_request_bytes:\n                    state.put("data", _stream_error_bytes("Ollama returned an oversized error response."))\n                    return\n            state.put("data", _stream_upstream_error(bytes(error_body), response.status))\n            return\n        if not streaming:\n            if not state.put("headers", (response.status, _stream_forward_headers(response.headers))):\n                return\n        if method != "HEAD":\n            pending = bytearray()\n            terminal = False\n            while not state.cancelled.is_set():\n                # read(65536) can wait for the entire buffer. read1 emits the\n                # currently available bytes, preserving each model flush.\n                chunk = response.read1(65536)\n                if not chunk:\n                    break\n                if not streaming:\n                    if not state.put("data", chunk):\n                        return\n                    continue\n                # A TCP read can end halfway through a JSON record. Emit only\n                # complete lines so a heartbeat never splits a JSON string.\n                pending.extend(chunk)\n                while True:\n                    newline = pending.find(b"\\n")\n                    if newline < 0:\n                        break\n                    if newline + 1 > server.max_request_bytes:\n                        state.put("data", _stream_error_bytes("Ollama returned an oversized NDJSON record."))\n                        return\n                    record = bytes(pending[:newline + 1])\n                    del pending[:newline + 1]\n                    if not state.put("data", record):\n                        return\n                    record_terminal = _stream_terminal_record(record)\n                    if record_terminal is not None:\n                        terminal = record_terminal\n                if len(pending) > server.max_request_bytes:\n                    state.put("data", _stream_error_bytes("Ollama returned an oversized NDJSON record."))\n                    return\n            if streaming and pending:\n                record = bytes(pending) + b"\\n"\n                if not state.put("data", record):\n                    return\n                record_terminal = _stream_terminal_record(record)\n                if record_terminal is not None:\n                    terminal = record_terminal\n            if streaming and not terminal and not state.cancelled.is_set():\n                state.put("data", _stream_error_bytes(\n                    "The model stream ended before completion. Please retry.", "upstream_eof"))\n    except Exception as error:\n        if not state.cancelled.is_set():\n            # Exception messages can include URLs/request data. Only report the\n            # exception type, and never log request/response content here.\n            state.put("error", _stream_error_bytes("Ollama connection failed (%s)." % type(error).__name__))\n    finally:\n        try:\n            if response is not None:\n                response.close()\n        finally:\n            connection.close()\n            state.put("done")\n            state.finished.set()\n\n\nclass _StreamProxyServer(_stream_server.ThreadingHTTPServer):\n    daemon_threads = True\n    allow_reuse_address = True\n\n    def __init__(self, address, upstream, heartbeat_seconds, heartbeat_padding_bytes, max_request_bytes):\n        self.upstream_host = upstream.hostname\n        self.upstream_port = upstream.port or 80\n        self.heartbeat_seconds = heartbeat_seconds\n        self.heartbeat = b" " * max(0, heartbeat_padding_bytes - 1) + b"\\n"\n        self.max_request_bytes = max_request_bytes\n        self.active_requests = set()\n        self.active_lock = _stream_threading.Lock()\n        super().__init__(address, _StreamProxyHandler)\n\n    def cancel_active(self):\n        with self.active_lock:\n            active = tuple(self.active_requests)\n        for request in active:\n            request.cancel()\n\n    def shutdown(self):\n        self.cancel_active()\n        super().shutdown()\n\n    def server_close(self):\n        self.cancel_active()\n        super().server_close()\n\n    def handle_error(self, request, client_address):\n        # BaseHTTPRequestHandler already handles client errors below. Keep the\n        # notebook log free of request targets and model text on all paths.\n        pass\n\n\nclass _StreamBodyError(Exception):\n    def __init__(self, status, message):\n        self.status = status\n        self.message = message\n        super().__init__(message)\n\n\nclass _StreamProxyHandler(_stream_server.BaseHTTPRequestHandler):\n    protocol_version = "HTTP/1.1"\n    server_version = "OllamaStreamProxy/1"\n\n    def log_message(self, *args):\n        pass\n\n    def _json_error(self, status, message):\n        body = _stream_error_bytes(message)\n        self.send_response(status)\n        self.send_header("Content-Type", "application/json")\n        self.send_header("Content-Length", str(len(body)))\n        self.send_header("Connection", "close")\n        self.end_headers()\n        if self.command != "HEAD":\n            self.wfile.write(body)\n        self.wfile.flush()\n        self.close_connection = True\n\n    def _chunk(self, body):\n        self.wfile.write(("%x\\r\\n" % len(body)).encode("ascii"))\n        self.wfile.write(body)\n        self.wfile.write(b"\\r\\n")\n        self.wfile.flush()\n\n    def _disconnected(self):\n        try:\n            readable, _, _ = _stream_select.select([self.connection], [], [], 0)\n            if readable:\n                # A clean TCP FIN closes only the peer\'s request writer. A\n                # tunnel can still be reading the response, so EOF alone must\n                # not cancel Ollama. Resets fail recv; fully closed readers are\n                # detected by the next response write/heartbeat instead.\n                self.connection.recv(1, _stream_socket.MSG_PEEK)\n            return False\n        except (OSError, ValueError):\n            return True\n\n    def _request_body(self):\n        lengths = self.headers.get_all("Content-Length", [])\n        encodings = self.headers.get_all("Transfer-Encoding", [])\n        if lengths and encodings:\n            raise _StreamBodyError(400, "Content-Length and Transfer-Encoding cannot be combined.")\n        if encodings:\n            if len(encodings) != 1 or encodings[0].strip().lower() != "chunked":\n                raise _StreamBodyError(501, "Only chunked request transfer encoding is supported.")\n            body = bytearray()\n            metadata_bytes = 0\n            while True:\n                line = self.rfile.readline(8193)\n                metadata_bytes += len(line)\n                if len(line) > 8192 or not line.endswith(b"\\r\\n") or metadata_bytes > 65536:\n                    raise _StreamBodyError(400, "Invalid or oversized chunk framing.")\n                size_text = line[:-2].split(b";", 1)[0]\n                if not size_text or len(size_text) > 16 or any(character not in b"0123456789abcdefABCDEF" for character in size_text):\n                    raise _StreamBodyError(400, "Invalid chunk size.")\n                size = int(size_text, 16)\n                if len(body) + size > self.server.max_request_bytes:\n                    raise _StreamBodyError(413, "Request body exceeds the proxy size limit.")\n                if size == 0:\n                    while True:\n                        trailer = self.rfile.readline(8193)\n                        metadata_bytes += len(trailer)\n                        if len(trailer) > 8192 or not trailer.endswith(b"\\r\\n") or metadata_bytes > 65536:\n                            raise _StreamBodyError(400, "Invalid or oversized chunk trailers.")\n                        if trailer == b"\\r\\n":\n                            return bytes(body)\n                        if (b":" not in trailer or trailer[:1] in (b" ", b"\\t")\n                                or trailer.split(b":", 1)[0].lower() in (b"content-length", b"transfer-encoding", b"host")):\n                            raise _StreamBodyError(400, "Invalid chunk trailer.")\n                chunk = self.rfile.read(size)\n                if len(chunk) != size or self.rfile.read(2) != b"\\r\\n":\n                    raise _StreamBodyError(400, "Incomplete chunked request body.")\n                body.extend(chunk)\n        if (len(lengths) > 1 or (lengths and (not lengths[0].strip().isascii()\n                or not lengths[0].strip().isdigit()))):\n            raise _StreamBodyError(400, "Invalid Content-Length.")\n        if lengths and len(lengths[0].strip()) > 16:\n            raise _StreamBodyError(413, "Request body exceeds the proxy size limit.")\n        length = int(lengths[0]) if lengths else 0\n        if length > self.server.max_request_bytes:\n            raise _StreamBodyError(413, "Request body exceeds the proxy size limit.")\n        body = self.rfile.read(length) if length else b""\n        if len(body) != length:\n            raise _StreamBodyError(400, "Incomplete request body.")\n        return body\n\n    def _dispatch(self):\n        self.close_connection = True\n        self.connection.settimeout(30)\n        try:\n            target = _stream_url.urlsplit(self.path)\n        except ValueError:\n            self._json_error(404, "Only local /api/* requests are supported.")\n            return\n        decoded_path = _stream_url.unquote(target.path)\n        if (target.scheme or target.netloc or target.fragment or not target.path.startswith("/api/")\n                or "\\\\" in decoded_path or any(part in (".", "..") for part in decoded_path.split("/"))\n                or any(ord(character) < 32 or ord(character) == 127 for character in decoded_path)):\n            self._json_error(404, "Only local /api/* requests are supported.")\n            return\n        try:\n            body = self._request_body()\n        except TimeoutError:\n            self._json_error(408, "Timed out reading request body.")\n            return\n        except _StreamBodyError as error:\n            self._json_error(error.status, error.message)\n            return\n        streaming = False\n        if self.command == "POST" and target.path in ("/api/chat", "/api/generate"):\n            try:\n                payload = _stream_json.loads(body)\n            except (ValueError, UnicodeError):\n                self._json_error(400, "Chat and generate requests require a JSON object.")\n                return\n            if not isinstance(payload, dict):\n                self._json_error(400, "Chat and generate requests require a JSON object.")\n                return\n            stream = payload.get("stream")\n            if stream is not None and not isinstance(stream, bool):\n                self._json_error(400, "The stream field must be true, false, or null.")\n                return\n            streaming = stream is not False\n        headers = dict(_stream_forward_headers(self.headers, request=True))\n        if body or self.command in ("POST", "PUT", "PATCH"):\n            headers["Content-Length"] = str(len(body))\n        if streaming:\n            headers["Accept-Encoding"] = "identity"\n        headers["Connection"] = "close"\n        state = _StreamRequest()\n        with self.server.active_lock:\n            self.server.active_requests.add(state)\n        response_started = False\n        try:\n            if streaming:\n                # Headers and a padded blank line arrive before Ollama receives\n                # the prompt. NDJSON parsers ignore these whitespace-only lines.\n                self.send_response(200)\n                self.send_header("Content-Type", "application/x-ndjson")\n                self.send_header("Transfer-Encoding", "chunked")\n                self.send_header("Cache-Control", "no-cache, no-transform")\n                self.send_header("X-Accel-Buffering", "no")\n                self.send_header("Connection", "close")\n                self.end_headers()\n                self._chunk(self.server.heartbeat)\n                response_started = True\n            state.thread = _stream_threading.Thread(\n                target=_stream_run_upstream,\n                args=(self.server, state, self.command, self.path, headers, body, streaming),\n                name="ollama-proxy-upstream", daemon=True,\n            )\n            state.thread.start()\n            next_heartbeat = _stream_time.monotonic() + self.server.heartbeat_seconds\n            while not state.cancelled.is_set():\n                if self._disconnected():\n                    break\n                wait = min(0.1, max(0, next_heartbeat - _stream_time.monotonic())) if streaming else 0.1\n                try:\n                    kind, value = state.events.get(timeout=wait)\n                except _stream_queue.Empty:\n                    if streaming and _stream_time.monotonic() >= next_heartbeat:\n                        self._chunk(self.server.heartbeat)\n                        next_heartbeat = _stream_time.monotonic() + self.server.heartbeat_seconds\n                    continue\n                if kind == "headers":\n                    status, response_headers = value\n                    self.send_response(status)\n                    for name, content in response_headers:\n                        self.send_header(name, content)\n                    self.send_header("Connection", "close")\n                    self.end_headers()\n                    self.wfile.flush()\n                    response_started = True\n                elif kind == "data":\n                    if streaming:\n                        self._chunk(value)\n                        next_heartbeat = _stream_time.monotonic() + self.server.heartbeat_seconds\n                    else:\n                        self.wfile.write(value)\n                        self.wfile.flush()\n                elif kind == "error":\n                    if streaming:\n                        self._chunk(value)\n                    elif not response_started:\n                        self._json_error(502, _stream_json.loads(value)["error"])\n                    else:\n                        break\n                elif kind == "done":\n                    if streaming:\n                        self.wfile.write(b"0\\r\\n\\r\\n")\n                        self.wfile.flush()\n                    break\n        except (OSError, ValueError):\n            # Includes clients cancelling while headers/body are being flushed.\n            pass\n        finally:\n            state.cancel()\n            with self.server.active_lock:\n                self.server.active_requests.discard(state)\n\n    do_GET = _dispatch\n    do_HEAD = _dispatch\n    do_POST = _dispatch\n    do_PUT = _dispatch\n    do_PATCH = _dispatch\n    do_DELETE = _dispatch\n    do_OPTIONS = _dispatch\n\n\ndef start_stream_proxy(upstream="http://127.0.0.1:11434", port=11435, heartbeat_seconds=10,\n                       heartbeat_padding_bytes=16384, max_request_bytes=4 * 1024 * 1024):\n    """Start a daemon HTTP server on 127.0.0.1 and return the server object.\n\n    Streamed chat/generate requests return HTTP 200 immediately. Upstream HTTP\n    errors therefore appear as NDJSON {"error": ...} records. Other API requests\n    preserve upstream status, headers and body. Padding is whitespace, not SSE.\n    heartbeat_padding_bytes=0 selects a single newline heartbeat. Stop using\n    server.shutdown() followed by server.server_close(); both cancel requests.\n    TCP resets cancel immediately; fully closed readers are detected at the next\n    response write/heartbeat because clean request-writer EOF is valid.\n    """\n    target = _stream_url.urlsplit(upstream)\n    if (target.scheme != "http" or target.hostname not in ("127.0.0.1", "localhost", "::1")\n            or target.username is not None or target.password is not None\n            or target.path not in ("", "/") or target.query or target.fragment):\n        raise ValueError("The streaming proxy requires a local HTTP upstream origin.")\n    if not isinstance(heartbeat_seconds, (int, float)) or not _stream_math.isfinite(heartbeat_seconds) or heartbeat_seconds <= 0:\n        raise ValueError("heartbeat_seconds must be positive and finite.")\n    if not isinstance(heartbeat_padding_bytes, int) or not 0 <= heartbeat_padding_bytes <= 1024 * 1024:\n        raise ValueError("heartbeat_padding_bytes must be between 0 and 1048576.")\n    if not isinstance(max_request_bytes, int) or not 1 <= max_request_bytes <= 64 * 1024 * 1024:\n        raise ValueError("max_request_bytes must be between 1 and 67108864.")\n    server = _StreamProxyServer(("127.0.0.1", port), target, heartbeat_seconds, heartbeat_padding_bytes, max_request_bytes)\n    server.thread = _stream_threading.Thread(target=server.serve_forever, name="ollama-stream-proxy", daemon=True)\n    server.thread.start()\n    return server\n\n\n"""Reconnectable Ollama jobs for the loopback stream proxy.\n\nThis module is bundled after kaggle-stream-proxy.py in the notebook namespace.\nCall install_async_job_routes(server) to extend that proxy explicitly.\n\nPOST /_jobs {"target":"/api/chat", "body":{...}} -> 202 {id, token, cursor:0}.\nOptional client-generated `job_id` (24+ URL-safe characters) and `token` (43+\nURL-safe characters) make start retries idempotent during retention. Generate\nboth cryptographically; the same identity/token must carry the same request.\nGET /_jobs/{id}/events?cursor=0&wait=20 -> bounded JSON with events, nextCursor,\ndone, status, upstreamStatus and error. Each event preserves the Ollama JSON in\nits `record` field. Cursor is the next event index and is safe to repeat. `done`\nmeans both the job and this cursor\'s backlog have finished; `jobDone` reports\nthe job state independently, so a completed multi-page result is fully drained.\nDELETE /_jobs/{id} cancels the origin socket. Read/cancel require\nAuthorization: Bearer <token>; tokens never belong in a query string.\nGET /_jobs/{id}/stream?cursor=0&max_seconds=120 -> bounded NDJSON leg with\n{cursor,record}, {cursor,heartbeat:true}, and terminal {cursor,done:true,...}.\nRecord and heartbeat envelopes include upstreamStatus when headers are known.\nRecord cursor is its event index; resume from record.cursor+1. Heartbeat and\nterminal cursors are already the next event index. EOF without done reconnects.\n\nPolling connections do not own/cancel the generation. Stop must call DELETE.\nCompleted jobs are retained for up to five minutes, but the oldest completed,\nfully closed jobs can be evicted earlier when another job needs capacity.\nA failed server-generated start-response write cancels that job. Client-chosen\nidentities remain retryable/cancellable even if the start reply is lost.\n"""\n\nimport hmac as _job_hmac\nimport hashlib as _job_hashlib\nimport http.client as _job_http\nimport json as _job_json\nimport math as _job_math\nimport secrets as _job_secrets\nimport threading as _job_threading\nimport time as _job_time\nimport urllib.parse as _job_url\n\n\nclass _JobAPIError(Exception):\n    def __init__(self, status, code, message):\n        self.status, self.code, self.message = status, code, message\n        super().__init__(message)\n\n\ndef _job_encode(value, sort_keys=False):\n    return _job_json.dumps(value, ensure_ascii=False, sort_keys=sort_keys, separators=(",", ":")).encode("utf-8")\n\n\ndef _job_member_bytes(raw, member):\n    """Extract a top-level JSON value\'s original UTF-8 lexical span.\n\n    The already-validated wrapper may contain nested objects, escaped quotes and\n    Unicode. JSONDecoder identifies boundaries; no inner value is reserialized.\n    Match json.loads\' last-value behavior for duplicate wrapper keys.\n    """\n    text, decoder = raw.decode("utf-8"), _job_json.JSONDecoder()\n\n    def whitespace(index):\n        while index < len(text) and text[index] in " \\t\\r\\n":\n            index += 1\n        return index\n\n    index, result = whitespace(0), None\n    if text[index:index + 1] != "{":\n        raise ValueError("Expected a job object")\n    index = whitespace(index + 1)\n    while text[index:index + 1] != "}":\n        key, index = decoder.raw_decode(text, index)\n        index = whitespace(index)\n        if text[index:index + 1] != ":":\n            raise ValueError("Expected a job member")\n        start = whitespace(index + 1)\n        _, end = decoder.raw_decode(text, start)\n        if key == member:\n            result = text[start:end].encode("utf-8")\n        index = whitespace(end)\n        if text[index:index + 1] == ",":\n            index = whitespace(index + 1)\n        elif text[index:index + 1] != "}":\n            raise ValueError("Expected a job object boundary")\n    if result is None:\n        raise ValueError("Missing job member")\n    return result\n\n\nclass _AsyncOllamaJob:\n    def __init__(self, manager, target, body, job_id=None, token=None):\n        self.manager, self.target, self.body = manager, target, body\n        self.id = job_id or _job_secrets.token_urlsafe(18)\n        self.token = token or _job_secrets.token_urlsafe(32)\n        self.client_identity = job_id is not None\n        self.request_hash = _job_hashlib.sha256(target.encode("ascii") + b"\\0" + body).digest()\n        self.condition = _job_threading.Condition()\n        self.request = _StreamRequest()\n        self.events = []\n        self.encoded_events = []\n        self.event_sizes = []\n        self.output_bytes = 0\n        self.created = self.last_poll = _job_time.monotonic()\n        self.finished_at = None\n        self.status, self.done, self.error = "running", False, None\n        self.upstream_status = None\n        self.thread = _job_threading.Thread(target=self.run, name="ollama-async-job", daemon=True)\n\n    def authenticated(self, token):\n        return (isinstance(token, str) and len(token) <= 128\n                and _job_hmac.compare_digest(token.encode("utf-8"), self.token.encode("ascii")))\n\n    def finish(self, status, error=None):\n        with self.condition:\n            if self.done:\n                return False\n            self.done, self.status, self.error = True, status, error\n            self.finished_at = _job_time.monotonic()\n            self.condition.notify_all()\n            return True\n\n    def fail(self, code, message):\n        self.finish("failed", {"code": code, "message": message})\n\n    def cancel(self, code="cancelled", message="The model job was cancelled."):\n        if self.finish("cancelled", {"code": code, "message": message}):\n            self.request.cancel()\n            return True\n        return False\n\n    def append(self, value, terminal=True, field="record", raw_record=None):\n        with self.condition:\n            if self.done or self.request.cancelled.is_set():\n                return False\n            event = {"cursor": len(self.events)}\n            if self.upstream_status is not None:\n                event["upstreamStatus"] = self.upstream_status\n            event[field] = value\n            if raw_record is not None and field == "record":\n                metadata = {name: part for name, part in event.items() if name != "record"}\n                encoded = _job_encode(metadata)[:-1] + b\',"record":\' + raw_record.strip() + b"}"\n            else:\n                encoded = _job_encode(event)\n            size = len(encoded)\n            if (size + 1 > self.manager.max_poll_bytes - 4096\n                    or self.output_bytes + size > self.manager.max_output_bytes\n                    or len(self.events) >= self.manager.max_output_events):\n                self.fail("output_limit", "The model job exceeded its output limit.")\n                return False\n            self.events.append(event)\n            self.encoded_events.append(encoded)\n            self.event_sizes.append(size)\n            self.output_bytes += size\n            self.condition.notify_all()\n            if terminal and isinstance(value, dict):\n                if "error" in value:\n                    self.fail("upstream_error", "Ollama returned an error record.")\n                    return False\n                if value.get("done") is True:\n                    self.finish("completed")\n                    return False\n            return True\n\n    def record(self, raw):\n        if not raw.strip():\n            return True\n        if len(raw) > self.manager.max_record_bytes:\n            self.fail("record_limit", "Ollama returned an oversized JSON record.")\n            return False\n        try:\n            value = _job_json.loads(raw)\n            return self.append(value, raw_record=raw)\n        except (ValueError, UnicodeError, RecursionError):\n            self.fail("upstream_invalid_json", "Ollama returned an invalid JSON record.")\n            return False\n\n    def run(self):\n        connection, response = None, None\n        try:\n            connection = _job_http.HTTPConnection(self.manager.server.upstream_host,\n                                                   self.manager.server.upstream_port, timeout=10)\n            if not self.request.set_connection(connection):\n                return\n            connection.connect()\n            if not self.request.set_socket(connection.sock):\n                return\n            connection.sock.settimeout(None)\n            if self.request.cancelled.is_set():\n                return\n            connection.request("POST", self.target, body=self.body, headers={\n                "Content-Type": "application/json", "Accept-Encoding": "identity", "Connection": "close",\n            })\n            response = connection.getresponse()\n            with self.condition:\n                self.upstream_status = response.status\n                self.condition.notify_all()\n            if not 200 <= response.status < 300:\n                error_body = bytearray()\n                while not self.request.cancelled.is_set():\n                    chunk = response.read1(min(65536, self.manager.max_record_bytes + 1 - len(error_body)))\n                    if not chunk:\n                        break\n                    error_body.extend(chunk)\n                    if len(error_body) > self.manager.max_record_bytes:\n                        self.fail("record_limit", "Ollama returned an oversized error response.")\n                        return\n                if self.request.cancelled.is_set():\n                    return\n                try:\n                    value = _job_json.loads(error_body)\n                    raw = bytes(error_body).strip()\n                    self.append(value, terminal=False, raw_record=raw if b"\\n" not in raw and b"\\r" not in raw else None)\n                except (ValueError, UnicodeError, RecursionError):\n                    self.append({"status": response.status, "body": bytes(error_body).decode("utf-8", "replace")},\n                                terminal=False, field="upstreamError")\n                self.fail("upstream_http_error", "Ollama returned HTTP %s." % response.status)\n                return\n            pending = bytearray()\n            while not self.request.cancelled.is_set():\n                chunk = response.read1(65536)\n                if not chunk:\n                    break\n                pending.extend(chunk)\n                while True:\n                    newline = pending.find(b"\\n")\n                    if newline < 0:\n                        break\n                    record = bytes(pending[:newline + 1])\n                    del pending[:newline + 1]\n                    if not self.record(record):\n                        return\n                if len(pending) > self.manager.max_record_bytes:\n                    self.fail("record_limit", "Ollama returned an oversized JSON record.")\n                    return\n            if self.request.cancelled.is_set():\n                return\n            if pending and not self.record(bytes(pending)):\n                return\n            self.fail("upstream_eof", "The model stream ended before completion. Please retry.")\n        except Exception as error:\n            if not self.request.cancelled.is_set():\n                self.fail("upstream_connection", "Ollama connection failed (%s)." % type(error).__name__)\n        finally:\n            # Own the connection until the worker actually exits. HTTP response\n            # and poll lifetimes never control this socket.\n            try:\n                self.request.cancel()\n            finally:\n                try:\n                    if response is not None:\n                        try:\n                            response.close()\n                        except OSError:\n                            pass\n                finally:\n                    try:\n                        if connection is not None:\n                            try:\n                                connection.close()\n                            except OSError:\n                                pass\n                    finally:\n                        self.body = b""  # Release prompt data after the job finishes.\n                        self.request.finished.set()\n\n    def poll(self, cursor, wait):\n        with self.condition:\n            self.last_poll = _job_time.monotonic()\n            if cursor > len(self.events):\n                raise _JobAPIError(409, "invalid_cursor", "The cursor is ahead of this job\'s events.")\n            deadline = _job_time.monotonic() + wait\n            while cursor == len(self.events) and not self.done:\n                remaining = deadline - _job_time.monotonic()\n                if remaining <= 0:\n                    break\n                self.condition.wait(remaining)\n            events, size, next_cursor = [], 4096, cursor\n            while next_cursor < len(self.events) and len(events) < self.manager.max_poll_events:\n                event_size = self.event_sizes[next_cursor] + 1\n                if size + event_size > self.manager.max_poll_bytes:\n                    break\n                events.append(self.events[next_cursor])\n                size += event_size\n                next_cursor += 1\n            return {\n                "id": self.id, "events": events, "nextCursor": next_cursor,\n                "done": self.done and next_cursor == len(self.events), "jobDone": self.done,\n                "status": self.status, "upstreamStatus": self.upstream_status,\n                "error": self.error, "hasMore": next_cursor < len(self.events),\n            }\n\n\nclass _AsyncOllamaJobs:\n    def __init__(self, server, max_running_jobs, max_retained_jobs, retention_seconds,\n                 idle_seconds, max_job_seconds, max_output_bytes, max_output_events,\n                 max_record_bytes, max_poll_bytes, max_poll_events, reaper_seconds):\n        self.server = server\n        self.max_running_jobs, self.max_retained_jobs = max_running_jobs, max_retained_jobs\n        self.retention_seconds, self.idle_seconds, self.max_job_seconds = retention_seconds, idle_seconds, max_job_seconds\n        self.max_output_bytes, self.max_output_events = max_output_bytes, max_output_events\n        self.max_record_bytes, self.max_poll_bytes, self.max_poll_events = max_record_bytes, max_poll_bytes, max_poll_events\n        self.reaper_seconds = reaper_seconds\n        self.jobs, self.lock = {}, _job_threading.Lock()\n        self.closed = _job_threading.Event()\n        self.thread = _job_threading.Thread(target=self.reap, name="ollama-job-reaper", daemon=True)\n        self.thread.start()\n\n    def start(self, target, body, job_id=None, token=None):\n        with self.lock:\n            if self.closed.is_set():\n                raise _JobAPIError(503, "shutting_down", "The job service is stopping.")\n            existing = self.jobs.get(job_id) if job_id is not None else None\n            if existing is not None:\n                if not existing.authenticated(token):\n                    raise _JobAPIError(403, "forbidden", "The job identity requires its original token.")\n                request_hash = _job_hashlib.sha256(target.encode("ascii") + b"\\0" + body).digest()\n                if not _job_hmac.compare_digest(existing.request_hash, request_hash):\n                    raise _JobAPIError(409, "job_conflict", "The job identity already belongs to another request.")\n                return existing, False\n            # Done is visible before its socket finishes closing. Release GPU\n            # capacity only after the owning upstream worker has actually exited.\n            running = sum(not job.request.finished.is_set() for job in self.jobs.values())\n            if running >= self.max_running_jobs:\n                raise _JobAPIError(429, "busy", "The model is already processing a job.")\n            # A busy conversation must not hit a five-minute rate limit after\n            # eight quick replies. Keep bounded replay history by evicting only\n            # the oldest completed jobs whose origin workers have fully exited.\n            if len(self.jobs) >= self.max_retained_jobs:\n                completed = sorted(\n                    (old for old in self.jobs.values() if old.done and old.request.finished.is_set()),\n                    key=lambda old: old.finished_at if old.finished_at is not None else old.created,\n                )\n                for old in completed:\n                    if len(self.jobs) < self.max_retained_jobs:\n                        break\n                    self.jobs.pop(old.id, None)\n            if len(self.jobs) >= self.max_retained_jobs:\n                raise _JobAPIError(429, "retention_limit", "The job service has reached its retention limit.")\n            job = _AsyncOllamaJob(self, target, body, job_id, token)\n            self.jobs[job.id] = job\n            try:\n                job.thread.start()\n            except Exception:\n                self.jobs.pop(job.id, None)\n                job.cancel()\n                job.request.finished.set()\n                raise\n            return job, True\n\n    def authenticate(self, job_id, token):\n        with self.lock:\n            job = self.jobs.get(job_id)\n        if job is None:\n            raise _JobAPIError(404, "not_found", "The job was not found or has expired.")\n        if not job.authenticated(token):\n            raise _JobAPIError(403, "forbidden", "A valid job bearer token is required.")\n        return job\n\n    def reap(self):\n        while not self.closed.wait(self.reaper_seconds):\n            now = _job_time.monotonic()\n            with self.lock:\n                jobs = tuple(self.jobs.values())\n            for job in jobs:\n                cancel_request = False\n                with job.condition:\n                    if not job.done:\n                        if now - job.created >= self.max_job_seconds:\n                            cancel_request = job.finish("cancelled", {\n                                "code": "job_timeout", "message": "The model job exceeded its runtime limit."})\n                        elif now - job.last_poll >= self.idle_seconds:\n                            cancel_request = job.finish("cancelled", {\n                                "code": "idle_timeout", "message": "The model job expired without polling."})\n                    expired = (job.done and job.finished_at is not None and job.request.finished.is_set()\n                               and now - job.finished_at >= self.retention_seconds)\n                if cancel_request:\n                    job.request.cancel()\n                if expired:\n                    with self.lock:\n                        if self.jobs.get(job.id) is job:\n                            self.jobs.pop(job.id, None)\n\n    def active_generations(self):\n        """Only active OpenCode job IDs can renew a relay generation lease."""\n        with self.lock:\n            ids = [job.id for job in self.jobs.values() if not job.request.finished.is_set()]\n        result = []\n        for job_id in ids:\n            owner, _, phase = job_id.rpartition("-")\n            parts = owner.split("-")\n            if (phase in ("main", "finish") and [len(part) for part in parts] == [8, 4, 4, 4, 12]\n                    and all(char in "0123456789abcdef" for part in parts for char in part)):\n                result.append(owner)\n        return sorted(set(result))[:8]\n\n    def cancel_generations(self, owners):\n        if not isinstance(owners, list):\n            return\n        selected = set(owner for owner in owners[:8] if isinstance(owner, str))\n        with self.lock:\n            jobs = tuple(self.jobs.values())\n        for job in jobs:\n            if job.id.rpartition("-")[0] in selected:\n                job.cancel()\n\n    def busy(self):\n        with self.lock:\n            return any(not job.request.finished.is_set() for job in self.jobs.values())\n\n    def close(self):\n        self.closed.set()\n        with self.lock:\n            jobs = tuple(self.jobs.values())\n        for job in jobs:\n            job.cancel("server_stopped", "The job service stopped.")\n\n\ndef install_async_job_routes(server, max_running_jobs=1, max_retained_jobs=8,\n                             retention_seconds=300, idle_seconds=120, max_job_seconds=3600,\n                             max_output_bytes=8 * 1024 * 1024, max_output_events=32768,\n                             max_record_bytes=1024 * 1024, max_poll_bytes=2 * 1024 * 1024,\n                             max_poll_events=256, reaper_seconds=1, stream_heartbeat_seconds=2,\n                             stream_padding_bytes=16384, max_stream_seconds=120, control_token=None):\n    """Explicitly extend an existing stream proxy; return its async job manager.\n\n    Retention is measured from completion; polling does not extend that limit.\n    Under capacity pressure the oldest completed/closed jobs can expire earlier.\n    An unfinished job must be polled within idle_seconds or it is cancelled.\n    All routes use no-store JSON responses and long polls are capped at 20s.\n    Streaming legs default to 120s; a personal coding server may opt into 300s.\n    Existing /api/* handling and its streaming behavior are inherited unchanged.\n    """\n    if hasattr(server, "_async_jobs"):\n        raise ValueError("Async job routes are already installed.")\n    for value in (max_running_jobs, max_retained_jobs, max_output_bytes, max_output_events,\n                  max_record_bytes, max_poll_bytes, max_poll_events):\n        if not isinstance(value, int) or isinstance(value, bool) or value <= 0:\n            raise ValueError("Job capacities must be positive integers.")\n    if max_running_jobs > max_retained_jobs or max_poll_bytes < 8192 or max_record_bytes + 4096 > max_poll_bytes:\n        raise ValueError("Inconsistent job capacity or polling limits.")\n    for value in (retention_seconds, idle_seconds, max_job_seconds, reaper_seconds, stream_heartbeat_seconds):\n        if not isinstance(value, (int, float)) or not _job_math.isfinite(value) or value <= 0:\n            raise ValueError("Job time limits must be positive and finite.")\n    if not isinstance(stream_padding_bytes, int) or not 0 <= stream_padding_bytes <= 1024 * 1024:\n        raise ValueError("Invalid stream heartbeat padding size.")\n    if (not isinstance(max_stream_seconds, (int, float)) or isinstance(max_stream_seconds, bool)\n            or not _job_math.isfinite(max_stream_seconds) or not 0 < max_stream_seconds <= 300):\n        raise ValueError("Invalid maximum stream leg duration.")\n    if control_token is not None and (not isinstance(control_token, str) or len(control_token) != 64\n            or any(char not in "0123456789abcdef" for char in control_token)):\n        raise ValueError("Invalid personal job-control credential.")\n    manager = _AsyncOllamaJobs(server, max_running_jobs, max_retained_jobs, retention_seconds,\n                              idle_seconds, max_job_seconds, max_output_bytes, max_output_events,\n                              max_record_bytes, max_poll_bytes, max_poll_events, reaper_seconds)\n    base_handler = server.RequestHandlerClass\n\n    class _AsyncJobHandler(base_handler):\n        def _job_json_response(self, status, payload, body=None):\n            body = _job_encode(payload) if body is None else body\n            self.send_response(status)\n            self.send_header("Content-Type", "application/json")\n            self.send_header("Content-Length", str(len(body)))\n            self.send_header("Cache-Control", "no-store")\n            self.send_header("Connection", "close")\n            self.end_headers()\n            if self.command != "HEAD":\n                self.wfile.write(body)\n            self.wfile.flush()\n            self.close_connection = True\n\n        def _job_token(self):\n            value = self.headers.get("Authorization", "")\n            return value[7:] if value.startswith("Bearer ") else ""\n\n        def _job_stream(self, job, cursor, max_seconds):\n            # Validate the cursor before committing HTTP 200 headers.\n            initial = job.poll(cursor, 0)\n            self.send_response(200)\n            self.send_header("Content-Type", "application/x-ndjson")\n            self.send_header("Transfer-Encoding", "chunked")\n            self.send_header("Cache-Control", "no-store, no-transform")\n            self.send_header("X-Accel-Buffering", "no")\n            self.send_header("Connection", "close")\n            self.end_headers()\n            deadline = _job_time.monotonic() + max_seconds\n\n            def send_record(record):\n                remaining = deadline - _job_time.monotonic()\n                if remaining <= 0:\n                    return False\n                self.connection.settimeout(min(30, remaining))\n                self._chunk(record)\n                return True\n\n            def heartbeat():\n                metadata = {"cursor": cursor, "heartbeat": True}\n                with job.condition:\n                    if job.upstream_status is not None:\n                        metadata["upstreamStatus"] = job.upstream_status\n                record = _job_encode(metadata)\n                return send_record(record + b" " * max(0, stream_padding_bytes - len(record) - 1) + b"\\n")\n\n            # Known failures can now reach the Worker before its raw response\n            # commits 200. Pending prefill still gets an immediate keepalive.\n            if not initial["jobDone"]:\n                heartbeat()\n            next_heartbeat = _job_time.monotonic() + stream_heartbeat_seconds\n            leg_ended = False\n            while _job_time.monotonic() < deadline:\n                wait = max(0, min(deadline, next_heartbeat) - _job_time.monotonic())\n                result = job.poll(cursor, wait)\n                for event in result["events"]:\n                    if not send_record(job.encoded_events[event["cursor"]] + b"\\n"):\n                        leg_ended = True\n                        break\n                    cursor = event["cursor"] + 1\n                    # Keep the padding cadence independent of token traffic. A\n                    # buffering tunnel otherwise receives only tiny records for\n                    # the whole leg and releases them together at its boundary.\n                    if _job_time.monotonic() >= next_heartbeat:\n                        if not heartbeat():\n                            leg_ended = True\n                            break\n                        next_heartbeat = _job_time.monotonic() + stream_heartbeat_seconds\n                if leg_ended:\n                    break\n                if result["done"]:\n                    send_record(_job_encode({"cursor": cursor, "done": True, "status": result["status"],\n                                             "upstreamStatus": result["upstreamStatus"], "error": result["error"]}) + b"\\n")\n                    break\n                if _job_time.monotonic() >= next_heartbeat:\n                    if not heartbeat():\n                        break\n                    next_heartbeat = _job_time.monotonic() + stream_heartbeat_seconds\n            # A normal leg boundary intentionally has no done record. Reattach\n            # with the next cursor; this does not cancel or replace the job.\n            self.wfile.write(b"0\\r\\n\\r\\n")\n            self.wfile.flush()\n\n        def _job_wait_idle(self, seconds):\n            self.send_response(200)\n            self.send_header("Content-Type", "application/x-ndjson")\n            self.send_header("Transfer-Encoding", "chunked")\n            self.send_header("Cache-Control", "no-store, no-transform")\n            self.send_header("X-Accel-Buffering", "no")\n            self.send_header("Connection", "close")\n            self.end_headers()\n            deadline = _job_time.monotonic() + seconds\n            next_heartbeat = 0\n            while True:\n                busy = manager.busy()\n                stamp = _job_time.monotonic()\n                if not busy or stamp >= next_heartbeat or stamp >= deadline:\n                    raw = _job_encode({"busy": busy})\n                    self.connection.settimeout(30)\n                    self._chunk(raw + b" " * max(0, stream_padding_bytes - len(raw) - 1) + b"\\n")\n                    next_heartbeat = stamp + stream_heartbeat_seconds\n                if not busy or stamp >= deadline:\n                    break\n                manager.closed.wait(min(0.05, max(0, deadline - stamp)))\n            self.wfile.write(b"0\\r\\n\\r\\n")\n            self.wfile.flush()\n\n        def _job_dispatch(self):\n            try:\n                target = _job_url.urlsplit(self.path)\n            except ValueError:\n                return base_handler._dispatch(self)\n            if not target.path.startswith("/_jobs"):\n                return base_handler._dispatch(self)\n            self.close_connection = True\n            self.connection.settimeout(30)\n            started_job = None\n            try:\n                if target.scheme or target.netloc or target.fragment:\n                    raise _JobAPIError(404, "not_found", "The job route was not found.")\n                if target.path == "/_jobs/control":\n                    if control_token is None:\n                        raise _JobAPIError(404, "not_found", "The control route is unavailable.")\n                    supplied = self._job_token()\n                    if (len(supplied) != 64 or any(char not in "0123456789abcdef" for char in supplied)\n                            or not _job_hmac.compare_digest(supplied, control_token)):\n                        raise _JobAPIError(403, "forbidden", "A valid personal control credential is required.")\n                    query = _job_url.parse_qs(target.query, keep_blank_values=True)\n                    try:\n                        if self.command != "GET" or set(query) != {"max_seconds"} or len(query["max_seconds"]) != 1:\n                            raise ValueError()\n                        seconds = float(query["max_seconds"][0])\n                        if not _job_math.isfinite(seconds) or not 0 < seconds <= max_stream_seconds:\n                            raise ValueError()\n                    except ValueError:\n                        raise _JobAPIError(400, "invalid_poll", "Supply a bounded control wait.")\n                    self._job_wait_idle(seconds)\n                    return\n                if target.path == "/_jobs" and self.command == "POST" and not target.query:\n                    try:\n                        raw_request = self._request_body()\n                        payload = _job_json.loads(raw_request)\n                    except _StreamBodyError as error:\n                        raise _JobAPIError(error.status, "invalid_request", error.message)\n                    except (ValueError, UnicodeError, RecursionError):\n                        raise _JobAPIError(400, "invalid_request", "A JSON job request is required.")\n                    if (not isinstance(payload, dict) or payload.get("target") not in ("/api/chat", "/api/generate")\n                            or not isinstance(payload.get("body"), dict)):\n                        raise _JobAPIError(400, "invalid_request", "Supply a chat/generate target and JSON body.")\n                    body = dict(payload["body"])\n                    if body.get("stream") is not None and body.get("stream") is not True:\n                        raise _JobAPIError(400, "invalid_request", "Async model jobs require streaming output.")\n                    job_id, token = payload.get("job_id"), payload.get("token")\n                    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789_-"\n                    if (job_id is not None or token is not None) and not (\n                            isinstance(job_id, str) and 24 <= len(job_id) <= 128 and all(char in alphabet for char in job_id)\n                            and isinstance(token, str) and 43 <= len(token) <= 128 and all(char in alphabet for char in token)):\n                        raise _JobAPIError(400, "invalid_identity", "Supply cryptographically random URL-safe job_id and token together.")\n                    try:\n                        # Validate Unicode, but forward the lexical body span.\n                        # In particular, int64 values and float spelling must\n                        # survive without a parse/stringify transport round trip.\n                        _job_encode(body)\n                        encoded_body = _job_member_bytes(raw_request, "body")\n                    except (ValueError, UnicodeError, RecursionError):\n                        raise _JobAPIError(400, "invalid_request", "The model body must contain valid JSON text.")\n                    job, created = manager.start(payload["target"], encoded_body, job_id, token)\n                    started_job = job if created else None\n                    self._job_json_response(202, {"id": job.id, "token": job.token,\n                                                   "cursor": 0, "maxWaitSeconds": 20})\n                    return\n                parts = target.path.split("/")\n                if len(parts) not in (3, 4) or parts[1] != "_jobs" or not parts[2]:\n                    raise _JobAPIError(404, "not_found", "The job route was not found.")\n                job = manager.authenticate(parts[2], self._job_token())\n                if self.command == "GET" and len(parts) == 4 and parts[3] in ("events", "stream"):\n                    query = _job_url.parse_qs(target.query, keep_blank_values=True)\n                    streaming = parts[3] == "stream"\n                    allowed = {"cursor", "max_seconds"} if streaming else {"cursor", "wait"}\n                    if set(query) - allowed or any(len(values) != 1 for values in query.values()):\n                        raise _JobAPIError(400, "invalid_poll", "Supply one cursor and wait value.")\n                    cursor_text = query.get("cursor", ["0"])[0]\n                    wait_text = query.get("max_seconds", ["120"])[0] if streaming else query.get("wait", ["15"])[0]\n                    try:\n                        if not cursor_text.isascii() or not cursor_text.isdigit() or len(cursor_text) > 12:\n                            raise ValueError()\n                        cursor, wait = int(cursor_text), float(wait_text)\n                        if not _job_math.isfinite(wait) or not (0 < wait <= max_stream_seconds if streaming else 0 <= wait <= 20):\n                            raise ValueError()\n                    except ValueError:\n                        raise _JobAPIError(400, "invalid_poll", "Cursor must be nonnegative and the requested wait within this route\'s limit.")\n                    if streaming:\n                        try:\n                            self._job_stream(job, cursor, wait)\n                        except OSError:\n                            # A leg\'s writer/timeout owns only that response.\n                            # Never turn its disconnect into job cancellation.\n                            pass\n                    else:\n                        result = job.poll(cursor, wait)\n                        metadata = dict(result)\n                        events = metadata.pop("events")\n                        encoded_events = [job.encoded_events[event["cursor"]] for event in events]\n                        encoded = _job_encode(metadata)[:-1] + b\',"events":[\' + b",".join(encoded_events) + b"]}"\n                        self._job_json_response(200, result, body=encoded)\n                elif self.command == "DELETE" and len(parts) == 3 and not target.query:\n                    cancelled = job.cancel()\n                    self._job_json_response(200, {"id": job.id, "cancelled": cancelled, "done": job.done})\n                else:\n                    raise _JobAPIError(405, "method_not_allowed", "Use POST to start, GET to poll, or DELETE to cancel.")\n            except _JobAPIError as error:\n                self._job_json_response(error.status, {"error": {"code": error.code, "message": error.message}})\n            except TimeoutError:\n                self._job_json_response(408, {"error": {"code": "request_timeout", "message": "The job request timed out."}})\n            except (OSError, UnicodeError):\n                # Poll readers can disconnect without changing the job. A start\n                # whose response cannot be delivered has no known owner.\n                if started_job is not None and not started_job.client_identity:\n                    started_job.cancel("start_disconnected", "The job start response could not be delivered.")\n\n        do_GET = _job_dispatch\n        do_HEAD = _job_dispatch\n        do_POST = _job_dispatch\n        do_PUT = _job_dispatch\n        do_PATCH = _job_dispatch\n        do_DELETE = _job_dispatch\n        do_OPTIONS = _job_dispatch\n\n    original_shutdown, original_close = server.shutdown, server.server_close\n\n    def _job_shutdown():\n        manager.close()\n        return original_shutdown()\n\n    def _job_close():\n        manager.close()\n        return original_close()\n\n    server.RequestHandlerClass = _AsyncJobHandler\n    server.shutdown, server.server_close, server._async_jobs = _job_shutdown, _job_close, manager\n    return manager\n\n\n# Heretic Qwen server for Kaggle (GPU T4 x2).\n#\n# The website\'s Start button pushes this script as a new version of a private Kaggle script and runs\n# it in the background. It reuses a pinned private dataset for Ollama, the tunnel and model files,\n# falling back to installation/downloads if needed. It loads the model with the largest context\n# that stays fully on the two GPUs, checks whether\n# it can see pictures, and registers the tunnel with the relay. Then it serves until the website asks\n# it to stop, nobody has used it for the idle time set in Settings, another server replaces it, or\n# Kaggle\'s 12-hour limit nears. Each step is reported to the website, which shows it while it starts.\n#\n# It can also make pictures with Qwen-Image 2.1 (ComfyUI, the official int8 files). The chat model and\n# the image model can\'t share the GPUs, so the website switches the server between the two; while it\n# chats, ComfyUI and the image files are prepared in the background so the first switch is quick.\n# Picture jobs are pulled from the relay and the results pushed back: ComfyUI only listens locally.\n#\n# The relay calls carry a launch token the Worker made for this run (it only lets this script\n# register, report progress, poll for commands and sign off, and expires after 13 hours), so the\n# relay key itself never leaves Cloudflare.\n\nimport base64\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport struct\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport zlib\n\nRELAY = \'\'\nLAUNCH_TOKEN = \'\'\nQUANT = \'Q6_K\'\nCACHE_ID = \'\'\nBATCH_SIZE = 128\nPROFILE_ID = \'colab-9b-q6-kv8-b128-32k\'\nFALLBACK_QUANT = "IQ3_M"\nREPO = "HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF"\nMODEL = "qwen3.8-hauhau"\nOLLAMA = "http://127.0.0.1:11434"\nSTREAM_PROXY = "http://127.0.0.1:11435"\nUSER_AGENT = "heretic-qwen-kaggle/1"\nOLLAMA_LOG = "/tmp/ollama.log"\nOLLAMA_BIN = "ollama"\nCLOUDFLARED_BIN = "cloudflared"\nOPLOG = None\n\nSTARTED = time.time()\n# Kaggle ends sessions at 12 hours; sign off cleanly a little before, so the page knows at once.\nMAX_RUNTIME_S = 11 * 3600 + 45 * 60\n# Tried largest first: the first that stays 100% on the GPUs wins (a bigger one spills to the CPU).\n# The Worker starts the list at the size that fit last time, so later starts skip the trial loads.\nCONTEXTS = [32768, 24576, 16384, 8192]\n# True once a run has seen this build answer about a picture: later runs skip the minute-long check.\nKNOWN_VISION = None\nAGENT_ONLY = False  # Personal coding sessions skip image setup and keep the exact model profile.\nRUNTIME_HOST = "kaggle"\nRELAY_HEADER = "x-launch-token"\nSTRICT_PROFILE = False  # Colab parity opts in: never lower precision/context or spill to CPU.\n# GPU use above this (percent, either GPU) counts as the model working on a reply.\nBUSY_GPU_PERCENT = 20\nDOWNLOAD_WORKERS = 16\nCHUNK_BYTES = 128 * 1024 * 1024\n# Polling: often enough that a switch to pictures starts within seconds, and quickly in image mode.\nCHAT_POLL_S = 10\nIMAGE_POLL_S = 2\nTUNNEL_CHECK_S = 30\n\nIMAGE_REPO = "abenzerps/Qwen-Image-2.1-Uncensored-Quantized"\nIMAGE_FILES = [\n    ("diffusion_models", "qwen-image-2.1-UC-int8_convrot.safetensors"),\n    ("text_encoders", "text_encoders/qwen3vl_8b_int8_convrot.safetensors"),\n    ("vae", "vae/qwen_image_2.1_vae_bf16.safetensors"),\n    ("loras", "qwen-image-2.1-uncensored-lora.safetensors"),\n]\nCOMFY_PORT = 8188\nCOMFY_LOG = "/tmp/comfyui.log"\nIMAGE_JOB_TIMEOUT_S = 30 * 60\n\n\ndef log(message):\n    print(time.strftime("%H:%M:%S"), message, flush=True)\n    if OPLOG is not None:\n        OPLOG.log(str(message).replace(REPO, "the model repository").replace(REPO.split("/")[0], "the model author"))\n\n\ndef public(text):\n    """Error text for the website: the model\'s repository owner is never shown there."""\n    return str(text).replace(REPO, "the model repository").replace(REPO.split("/")[0], "the model author")[:180]\n\n\ndef http(method, url, body=None, headers=None, timeout=60):\n    data = None if body is None else json.dumps(body).encode()\n    request = urllib.request.Request(\n        url,\n        data=data,\n        method=method,\n        headers={"content-type": "application/json", "user-agent": USER_AGENT, **(headers or {})},\n    )\n    with urllib.request.urlopen(request, timeout=timeout) as response:\n        raw = response.read()\n    return json.loads(raw) if raw else None\n\n\ndef relay(method, path, body=None):\n    return http(method, RELAY + path, body, {RELAY_HEADER: LAUNCH_TOKEN}, timeout=30)\n\n\ndef upload(path, data, content_type):\n    """Posts raw bytes to the relay (a finished picture)."""\n    request = urllib.request.Request(\n        RELAY + path, data=data, method="POST",\n        headers={"content-type": content_type, "user-agent": USER_AGENT, RELAY_HEADER: LAUNCH_TOKEN},\n    )\n    with urllib.request.urlopen(request, timeout=120) as response:\n        raw = response.read()\n    return json.loads(raw) if raw else None\n\n\ndef report(stage, detail=None, percent=None):\n    if OPLOG is not None:\n        OPLOG.log(" · ".join(str(value) for value in (stage, detail, f"{percent}%" if percent is not None else None) if value is not None))\n    try:\n        relay("POST", "/_relay/progress", {"host": RUNTIME_HOST, "stage": stage, "detail": detail, "percent": percent})\n    except Exception as err:  # the page just won\'t show this step\n        log(f"progress report failed: {err}")\n\n\nclass Background:\n    """Runs fn in a thread; result() waits for it and returns its value or raises its error."""\n\n    def __init__(self, fn):\n        self.value = None\n        self.error = None\n        self.thread = threading.Thread(target=self._run, args=(fn,), daemon=True)\n        self.thread.start()\n\n    def _run(self, fn):\n        try:\n            self.value = fn()\n        except BaseException as err:  # handed to whoever waits for the result\n            self.error = err\n\n    def result(self):\n        self.thread.join()\n        if self.error:\n            raise self.error\n        return self.value\n\n\ndef sh(command):\n    subprocess.run(command, shell=True, check=True)\n\n\ndef tail(path, lines=3):\n    try:\n        with open(path, encoding="utf-8", errors="replace") as f:\n            return " | ".join(f.read().strip().splitlines()[-lines:])\n    except OSError:\n        return ""\n\n\n# ---------------------------------------------------------------------------------------- set-up\n\n\ndef install(cache=None, runtime_root="/tmp/heretic-runtime"):\n    global OLLAMA_BIN, CLOUDFLARED_BIN, PROFILE_ID, KNOWN_VISION\n    if cache:\n        try:\n            report("installing", "Preparing the saved runtime")\n            runtime = install_cached_runtime(cache, runtime_root)\n            subprocess.run([runtime["ollama"], "--version"], check=True, capture_output=True, timeout=30)\n            subprocess.run([runtime["cloudflared"], "--version"], check=True, capture_output=True, timeout=30)\n            OLLAMA_BIN, CLOUDFLARED_BIN = runtime["ollama"], runtime["cloudflared"]\n            log("Runtime cache hit: Ollama and cloudflared; no runtime downloads")\n            return\n        except Exception as err:\n            log(f"Runtime cache unavailable ({public(err)}); installing normally")\n    # A downloaded runtime is not the measured immutable profile. Re-probe its fit and vision.\n    PROFILE_ID = ""\n    KNOWN_VISION = None\n    report("installing", "Installing Ollama and the tunnel")\n    if not shutil.which("ollama"):\n        log("Installing Ollama...")\n        try:\n            sh("(command -v zstd >/dev/null || (apt-get -qq update && apt-get -qq install -y zstd)) >/tmp/apt.log 2>&1")\n            sh("curl -fsSL https://ollama.com/install.sh | sh >/tmp/ollama-install.log 2>&1")\n        except subprocess.CalledProcessError:\n            raise RuntimeError("Installing Ollama failed: " + tail("/tmp/ollama-install.log"))\n    if not os.path.exists("/usr/local/bin/cloudflared"):\n        log("Installing cloudflared...")\n        sh(\n            "curl -fsSL -o /usr/local/bin/cloudflared "\n            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"\n            " && chmod +x /usr/local/bin/cloudflared"\n        )\n\n\ndef models_dir():\n    """Ollama\'s store on the disk with the most room (never /kaggle/working, which Kaggle saves)."""\n    roots = [d for d in ("/kaggle/tmp", "/tmp") if os.path.isdir(d)]\n    root = max(roots, key=lambda d: shutil.disk_usage(d).free)\n    path = os.path.join(root, "ollama-models")\n    os.makedirs(os.path.join(path, "blobs"), exist_ok=True)\n    return path, shutil.disk_usage(root).free\n\n\ndef start_ollama(models):\n    env = {\n        **os.environ,\n        "OLLAMA_HOST": "127.0.0.1:11434",\n        "OLLAMA_MODELS": models,\n        "OLLAMA_FLASH_ATTENTION": "1",\n        "OLLAMA_KV_CACHE_TYPE": "q8_0",\n        "OLLAMA_KEEP_ALIVE": "-1",\n        "OLLAMA_NUM_PARALLEL": "1",\n        "OLLAMA_MAX_LOADED_MODELS": "1",\n        # A cold read of the mounted 27B weights can pause longer than Ollama\'s default 5m.\n        # Keep stall detection bounded without treating slow storage as a smaller GPU fit.\n        "OLLAMA_LOAD_TIMEOUT": "2h" if AGENT_ONLY else "10m",\n        # Files are written into the store while the server runs; don\'t let it tidy them away.\n        "OLLAMA_NOPRUNE": "1",\n    }\n    subprocess.Popen([OLLAMA_BIN, "serve"], env=env, stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT)\n    for _ in range(60):\n        try:\n            http("GET", OLLAMA + "/api/version", timeout=5)\n            return\n        except Exception:\n            time.sleep(1)\n    raise RuntimeError("Ollama didn\'t start: " + tail(OLLAMA_LOG))\n\n\n# ------------------------------------------------------------------------------------ the model\n\n\ndef model_files(quant):\n    """The quantization\'s GGUF and the vision projector: (name, sha256, size) from the repo listing."""\n    tree = http("GET", f"https://huggingface.co/api/models/{REPO}/tree/main")\n    files = [f for f in tree if f.get("type") == "file" and f.get("lfs")]\n    model = next(f for f in files if f["path"].endswith(f"-{quant}.gguf"))\n    projector = next(f for f in files if f["path"].startswith("mmproj-") and f["path"].endswith(".gguf"))\n    return [(f["path"], f["lfs"]["oid"], f["lfs"]["size"]) for f in (model, projector)]\n\n\ndef signed_url(name, repo=REPO):\n    """Where the file actually downloads from (the CDN URL behind Hugging Face\'s redirect)."""\n    request = urllib.request.Request(\n        f"https://huggingface.co/{repo}/resolve/main/{name}",\n        headers={"range": "bytes=0-0", "user-agent": USER_AGENT},\n    )\n    with urllib.request.urlopen(request, timeout=60) as response:\n        return response.geturl()\n\n\ndef boot_progress(finished, total):\n    report("downloading", f"{finished / 1e9:.1f} of {total / 1e9:.1f} GB", int(finished * 100 / total))\n\n\ndef download(files, dest_for, repo=REPO, on_progress=boot_progress, workers=DOWNLOAD_WORKERS):\n    """\n    Downloads each file to dest_for(name, sha) in parallel byte ranges (far faster than one\n    stream), calling on_progress(finished, total) every 10 seconds.\n    """\n    total = sum(size for _, _, size in files)\n    finished = 0\n    lock = threading.Lock()\n    done = threading.Event()\n\n    def reporter():\n        while not done.wait(10):\n            with lock:\n                now = finished\n            on_progress(now, total)\n\n    threading.Thread(target=reporter, daemon=True).start()\n    try:\n        for name, sha, size in files:\n            dest = dest_for(name, sha)\n            if os.path.exists(dest) and os.path.getsize(dest) == size:\n                finished += size\n                continue\n            log(f"Downloading {name.replace(REPO.split(\'/\')[0], \'\').strip(\'-\')} ({size / 1e9:.1f} GB)...")\n            partial = dest + "-partial"\n            with open(partial, "wb") as f:\n                f.truncate(size)\n            url = [signed_url(name, repo)]\n            ranges = [(start, min(start + CHUNK_BYTES, size) - 1) for start in range(0, size, CHUNK_BYTES)]\n            errors = []\n\n            def worker():\n                nonlocal finished\n                while True:\n                    with lock:\n                        if not ranges or errors:\n                            return\n                        start, end = ranges.pop()\n                    for attempt in range(6):\n                        try:\n                            request = urllib.request.Request(url[0], headers={"range": f"bytes={start}-{end}", "user-agent": USER_AGENT})\n                            with urllib.request.urlopen(request, timeout=120) as response, open(partial, "r+b") as f:\n                                f.seek(start)\n                                written = 0\n                                while chunk := response.read(1 << 20):\n                                    f.write(chunk)\n                                    written += len(chunk)\n                            if written != end - start + 1:\n                                raise IOError(f"short read ({written} of {end - start + 1} bytes)")\n                            with lock:\n                                finished += written\n                            break\n                        except Exception as err:\n                            log(f"range {start}-{end} attempt {attempt + 1} failed: {err}")\n                            if isinstance(err, urllib.error.HTTPError) and err.code in (401, 403):\n                                url[0] = signed_url(name, repo)  # the signed link expired\n                            time.sleep(2 * (attempt + 1))\n                    else:\n                        with lock:\n                            errors.append(f"part of the model wouldn\'t download ({start}-{end})")\n                        return\n\n            threads = [threading.Thread(target=worker) for _ in range(workers)]\n            for t in threads:\n                t.start()\n            for t in threads:\n                t.join()\n            if errors:\n                raise RuntimeError(errors[0])\n            os.replace(partial, dest)\n    finally:\n        done.set()\n\n\ndef install_manifest(models, files, quant):\n    """\n    Registers the downloaded files with Ollama as MODEL, exactly the way `ollama pull` of the same\n    repository stores them (model + vision projector layers, Hugging Face\'s config), so the model\n    behaves the same as the Colab server\'s.\n    """\n    (_, model_sha, model_size), (_, projector_sha, projector_size) = files\n    try:\n        manifest = http(\n            "GET",\n            f"https://huggingface.co/v2/{REPO}/manifests/{FALLBACK_QUANT}",\n            headers={"accept": "application/vnd.docker.distribution.manifest.v2+json"},\n        )\n        config = http("GET", f"https://huggingface.co/v2/{REPO}/blobs/{manifest[\'config\'][\'digest\']}")\n    except Exception as err:\n        log(f"using a built-in model config ({err})")\n        config = {"model_format": "gguf", "model_family": "qwen35", "model_families": ["qwen35"], "architecture": "amd64", "os": "linux"}\n    config["file_type"] = quant\n    config["rootfs"] = {"type": "layers", "diff_ids": [f"sha256:{model_sha}", f"sha256:{projector_sha}"]}\n    raw = json.dumps(config, separators=(",", ":")).encode()\n    config_sha = hashlib.sha256(raw).hexdigest()\n    with open(os.path.join(models, "blobs", f"sha256-{config_sha}"), "wb") as f:\n        f.write(raw)\n    manifest = {\n        "schemaVersion": 2,\n        "mediaType": "application/vnd.docker.distribution.manifest.v2+json",\n        "config": {"mediaType": "application/vnd.docker.container.image.v1+json", "digest": f"sha256:{config_sha}", "size": len(raw)},\n        "layers": [\n            {"mediaType": "application/vnd.ollama.image.model", "digest": f"sha256:{model_sha}", "size": model_size},\n            {"mediaType": "application/vnd.ollama.image.projector", "digest": f"sha256:{projector_sha}", "size": projector_size},\n        ],\n    }\n    path = os.path.join(models, "manifests", "registry.ollama.ai", "library", MODEL)\n    os.makedirs(path, exist_ok=True)\n    with open(os.path.join(path, "latest"), "w") as f:\n        json.dump(manifest, f)\n    http("POST", OLLAMA + "/api/show", {"model": MODEL}, timeout=120)  # fails if Ollama can\'t read it\n\n\ndef pull_fallback():\n    """The smaller build through `ollama pull`, the way the Colab server gets it."""\n    source = f"hf.co/{REPO}:{FALLBACK_QUANT}"\n    request = urllib.request.Request(\n        OLLAMA + "/api/pull",\n        data=json.dumps({"model": source, "stream": True}).encode(),\n        headers={"content-type": "application/json"},\n    )\n    last = 0\n    with urllib.request.urlopen(request, timeout=3600) as response:\n        for line in response:\n            event = json.loads(line or b"{}")\n            if event.get("error"):\n                raise RuntimeError(public(event["error"]))\n            if event.get("total") and time.time() - last > 10:\n                last = time.time()\n                report("downloading", f"Smaller build: {event.get(\'completed\', 0) / 1e9:.1f} of {event[\'total\'] / 1e9:.1f} GB", int(event.get("completed", 0) * 100 / event["total"]))\n    http("POST", OLLAMA + "/api/copy", {"source": source, "destination": MODEL})\n\n\ndef fetch_model(models, free_bytes):\n    """Downloads the chosen build into Ollama\'s store (needs nothing installed, so it runs first)."""\n    files = model_files(QUANT)\n    needed = sum(size for _, _, size in files) + (1 << 30)\n    if free_bytes < needed:\n        raise RuntimeError(f"only {free_bytes / 1e9:.0f} GB of disk free, {needed / 1e9:.0f} GB needed")\n    blobs = os.path.join(models, "blobs")\n    download(files, lambda name, sha: os.path.join(blobs, "sha256-" + sha))\n    return files\n\n\ndef get_model(models, fetched):\n    """Registers the downloaded build with Ollama; the smaller one if anything about it failed."""\n    try:\n        install_manifest(models, fetched.result(), QUANT)\n        return QUANT\n    except Exception as err:\n        if CACHE_ID or AGENT_ONLY or STRICT_PROFILE:\n            raise RuntimeError(f"The configured {QUANT} model is unavailable ({public(err)}). Try starting again.") from err\n        log(f"{QUANT} unavailable ({public(err)}); using {FALLBACK_QUANT}")\n        report("downloading", f"Using the smaller build instead ({public(err)})", 0)\n        pull_fallback()\n        return FALLBACK_QUANT\n\n\ndef load(context):\n    """Loads the model at `context`; True when all of it is on the GPUs."""\n    mark = os.path.getsize(OLLAMA_LOG) if os.path.exists(OLLAMA_LOG) else 0\n    # Match website chat/compaction: changing context-shift later reloads Ollama\'s runner.\n    try:\n        http("POST", OLLAMA + "/api/generate", {"model": MODEL, "prompt": "", "stream": False, "keep_alive": -1, "shift": False, "options": {"num_ctx": context, "num_batch": BATCH_SIZE}}, timeout=7200 if AGENT_ONLY else 1800)\n    except urllib.error.HTTPError as err:\n        # Only this local, empty-prompt load\'s bounded JSON error is useful here.\n        # Do not dump an arbitrary response body, request, environment or tunnel URL.\n        detail = ""\n        try:\n            payload = json.loads(err.read(4096).decode("utf-8", "replace"))\n            if isinstance(payload, dict) and isinstance(payload.get("error"), str):\n                detail = payload["error"].replace(LAUNCH_TOKEN, "[redacted]")\n                detail = re.sub(r"https?://\\S+", "[URL hidden]", detail)\n                detail = public(re.sub(r"[\\r\\n\\t]+", " ", detail))\n        except (ValueError, OSError):\n            pass\n        finally:\n            err.close()\n        suffix = f": {detail}" if detail else ""\n        raise RuntimeError(f"Ollama model load failed (HTTP {err.code}){suffix}") from None\n    loaded = http("GET", OLLAMA + "/api/ps").get("models", [])\n    model = next((m for m in loaded if m.get("name", "").split(":")[0] == MODEL), None)\n    if not model:\n        return False\n    # How Ollama split the model\'s layers, from its own log (for the record: what spilled over).\n    try:\n        with open(OLLAMA_LOG, "rb") as f:\n            f.seek(mark)\n            layers = re.findall(r"offloaded (\\d+)/(\\d+) layers to GPU", f.read().decode("utf-8", "replace"))\n    except OSError:\n        layers = []\n    split = f", {layers[-1][0]}/{layers[-1][1]} layers on the GPUs" if layers else ""\n    log(f"{context} context: {model.get(\'size_vram\', 0) / 1e9:.1f} of {model.get(\'size\', 0) / 1e9:.1f} GB on the GPUs{split}")\n    return model.get("size_vram", 0) >= model.get("size", 1)\n\n\ndef load_best():\n    global PROFILE_ID\n    for context in CONTEXTS:\n        report("loading", f"Trying {context // 1024}K tokens of memory")\n        try:\n            if load(context):\n                return context\n        except Exception as err:\n            # A timeout or runner failure is not a measurement of the maximum GPU fit.\n            # The actual lower context still registers, but must not teach this profile\'s cap.\n            PROFILE_ID = ""\n            log(f"{context} context failed to load: {err}")\n    if AGENT_ONLY or STRICT_PROFILE:\n        raise RuntimeError("The selected Q5_K_P / 98,304-token profile did not fit entirely on the available GPUs. No smaller model or context was substituted.")\n    # Nothing fits entirely: the smallest still works, partly on the CPU.\n    load(CONTEXTS[-1])\n    return CONTEXTS[-1]\n\n\ndef red_square_png(size=32):\n    row = b"\\x00" + b"\\xff\\x00\\x00" * size\n    chunk = lambda kind, data: struct.pack(">I", len(data)) + kind + data + struct.pack(">I", zlib.crc32(kind + data) & 0xFFFFFFFF)\n    return (\n        b"\\x89PNG\\r\\n\\x1a\\n"\n        + chunk(b"IHDR", struct.pack(">IIBBBBB", size, size, 8, 2, 0, 0, 0))\n        + chunk(b"IDAT", zlib.compress(row * size))\n        + chunk(b"IEND", b"")\n    )\n\n\ndef sees_pictures(context):\n    report("loading", "Checking whether the model can see pictures")\n    try:\n        reply = http(\n            "POST",\n            OLLAMA + "/api/chat",\n            {\n                "model": MODEL,\n                "stream": False,\n                "think": False,\n                "keep_alive": -1,\n                "shift": False,\n                "messages": [{"role": "user", "content": "What colour fills this picture? Answer with one word.", "images": [base64.b64encode(red_square_png()).decode()]}],\n                "options": {"num_ctx": context, "num_batch": BATCH_SIZE, "num_predict": 16, "temperature": 0},\n            },\n            timeout=600,\n        )\n        answer = reply.get("message", {}).get("content", "")\n        log(f"picture check answered: {answer!r}")\n        return "red" in answer.lower()\n    except Exception as err:\n        log(f"picture check failed: {err}")\n        return False\n\n\n# ------------------------------------------------------------------------------- pictures\n\n\nclass Pictures:\n    """\n    ComfyUI with Qwen-Image 2.1. prepare() installs it and fetches the files in the background while\n    the server chats; start() and stop() hand it the GPUs and take them back; make() runs one job.\n    """\n\n    def __init__(self, models_root, cache=None):\n        self.dir = os.path.join(models_root, "ComfyUI")\n        self.cache = cache\n        self.status = "waiting"  # what prepare() is doing, for the page\n        self.percent = 0\n        self.ready = threading.Event()\n        self.error = None\n        self.process = None\n        self.prep = None\n\n    def prepare(self):\n        """Starts the background preparation once; safe to call again."""\n        if not self.prep:\n            self.prep = Background(self._prepare)\n\n    def _prepare(self):\n        try:\n            free = shutil.disk_usage(os.path.dirname(self.dir)).free\n            cached_images = None\n            if self.cache:\n                try:\n                    cached_images = validate_image_cache(self.cache, repo=IMAGE_REPO)\n                    if {(f["folder"], f["source_path"]) for f in cached_images["files"]} != set(IMAGE_FILES):\n                        raise CacheError("cached picture files do not match this server")\n                    stage_cached_images(self.cache, os.path.join(self.dir, "models"), repo=IMAGE_REPO)\n                    log("Picture model cache hit: all four files; no image-model downloads")\n                except Exception as err:\n                    cached_images = None\n                    log(f"Picture cache unavailable ({public(err)}); downloading normally")\n            files = [] if cached_images else image_files()\n            needed = sum(size for _, _, size, _ in files) + (3 << 30)\n            if free < needed:\n                raise RuntimeError(f"only {free / 1e9:.0f} GB of disk free for pictures, {needed / 1e9:.0f} GB needed")\n            self.status = "Installing the picture tools"\n            if not os.path.isdir(os.path.join(self.dir, "comfy")):\n                # Cache links may already occupy models/, so clone into a sibling then merge.\n                source = self.dir + "-source"\n                sh(f"git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI {source} >/tmp/comfy-install.log 2>&1")\n                shutil.copytree(source, self.dir, dirs_exist_ok=True)\n                shutil.rmtree(source)\n            # Kaggle\'s own torch stays: ComfyUI\'s pins would download gigabytes for nothing.\n            with open(os.path.join(self.dir, "requirements.txt")) as f:\n                wanted = [line.strip() for line in f if line.strip() and not line.startswith("#") and not re.match(r"torch(vision|audio)?\\b", line.strip())]\n            with open("/tmp/comfy-requirements.txt", "w") as f:\n                f.write("\\n".join(wanted) + "\\n")\n            sh("pip install -q -r /tmp/comfy-requirements.txt >>/tmp/comfy-install.log 2>&1")\n\n            def progress(finished, total):\n                self.status = f"Downloading the picture model: {finished / 1e9:.1f} of {total / 1e9:.1f} GB"\n                self.percent = int(finished * 100 / total)\n\n            dest = {name: os.path.join(self.dir, "models", folder, os.path.basename(name)) for folder, name, _, _ in files}\n            for path in dest.values():\n                os.makedirs(os.path.dirname(path), exist_ok=True)\n            if files:\n                download([(name, sha, size) for _, name, size, sha in files], lambda name, sha: dest[name], IMAGE_REPO, progress, workers=8)\n            self.status = "Ready"\n            self.percent = 100\n        except Exception as err:\n            self.error = f"{type(err).__name__}: {err}" if not isinstance(err, RuntimeError) else str(err)\n            log(f"preparing pictures failed: {self.error} | {tail(\'/tmp/comfy-install.log\')}")\n        finally:\n            self.ready.set()\n\n    def start(self):\n        """Starts ComfyUI (the chat model must be unloaded first) and waits until it answers."""\n        if self.process and self.process.poll() is None:\n            return\n        out = os.path.join(os.path.dirname(self.dir), "comfy-output")\n        os.makedirs(out, exist_ok=True)\n        self.process = subprocess.Popen(\n            [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT), "--disable-auto-launch", "--output-directory", out],\n            cwd=self.dir, stdout=open(COMFY_LOG, "a"), stderr=subprocess.STDOUT,\n        )\n        for _ in range(240):\n            if self.process.poll() is not None:\n                raise RuntimeError("ComfyUI stopped while starting: " + tail(COMFY_LOG, 4))\n            try:\n                http("GET", f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=5)\n                return\n            except Exception:\n                time.sleep(1)\n        raise RuntimeError("ComfyUI didn\'t start: " + tail(COMFY_LOG, 4))\n\n    def stop(self):\n        if self.process and self.process.poll() is None:\n            self.process.terminate()\n            try:\n                self.process.wait(timeout=30)\n            except subprocess.TimeoutExpired:\n                self.process.kill()\n        self.process = None\n\n    def make(self, job, keep_going):\n        """\n        Runs one job and uploads the result. keep_going(step) reports a step and answers False when\n        the page deleted the picture, which stops ComfyUI.\n        """\n        comfy = f"http://127.0.0.1:{COMFY_PORT}"\n        mark = os.path.getsize(COMFY_LOG) if os.path.exists(COMFY_LOG) else 0\n        try:\n            queued = http("POST", comfy + "/prompt", {"prompt": workflow(job), "client_id": "heretic-qwen"}, timeout=60)\n        except urllib.error.HTTPError as err:\n            raise RuntimeError("ComfyUI refused the job: " + err.read().decode("utf-8", "replace")[:300])\n        prompt_id = queued["prompt_id"]\n        started = time.time()\n        step = -1\n        while time.time() - started < IMAGE_JOB_TIMEOUT_S:\n            time.sleep(1.5)\n            history = http("GET", f"{comfy}/history/{prompt_id}", timeout=30).get(prompt_id)\n            if history:\n                status = history.get("status") or {}\n                if status.get("status_str") == "error":\n                    messages = [m[1].get("exception_message", "") for m in status.get("messages", []) if m and m[0] == "execution_error"]\n                    raise RuntimeError("ComfyUI failed: " + (messages[0] if messages else "unknown error")[:300])\n                images = [img for out in history.get("outputs", {}).values() for img in out.get("images", [])]\n                if images:\n                    return self._deliver(comfy, job, images[0])\n            now = sampling_step(mark)\n            if now != step:\n                step = now\n                if not keep_going(max(step, 0)):\n                    http("POST", comfy + "/interrupt", {}, timeout=10)\n                    http("POST", comfy + "/queue", {"delete": [prompt_id]}, timeout=10)\n                    log(f"picture {job[\'id\']} was deleted on the page; stopped it")\n                    return\n        http("POST", comfy + "/interrupt", {}, timeout=10)\n        raise RuntimeError("The picture took more than 30 minutes")\n\n    def _deliver(self, comfy, job, image):\n        query = urllib.parse.urlencode({"filename": image["filename"], "subfolder": image.get("subfolder", ""), "type": image.get("type", "output")})\n        with urllib.request.urlopen(f"{comfy}/view?{query}", timeout=120) as response:\n            png = response.read()\n        from io import BytesIO\n        from PIL import Image\n\n        picture = Image.open(BytesIO(png))\n        low, high = zip(*picture.convert("RGB").getextrema())\n        if max(h - l for l, h in zip(low, high)) < 4:\n            raise RuntimeError("The picture came out blank (one flat colour)")\n        preview = picture.convert("RGB")\n        preview.thumbnail((640, 640))\n        buffer = BytesIO()\n        preview.save(buffer, "WEBP", quality=82)\n        upload(f"/_relay/image-result?id={job[\'id\']}&kind=thumb", buffer.getvalue(), "image/webp")\n        upload(f"/_relay/image-result?id={job[\'id\']}&kind=full", png, "image/png")\n        try:\n            os.remove(os.path.join(os.path.dirname(self.dir), "comfy-output", image.get("subfolder", ""), image["filename"]))\n        except OSError:\n            pass\n\n\ndef image_files():\n    """(folder, name, size, sha256) for each picture file, from the repo listing."""\n    tree = http("GET", f"https://huggingface.co/api/models/{IMAGE_REPO}/tree/main?recursive=1")\n    by_path = {f["path"]: f for f in tree if f.get("type") == "file" and f.get("lfs")}\n    return [(folder, name, by_path[name]["lfs"]["size"], by_path[name]["lfs"]["oid"]) for folder, name in IMAGE_FILES]\n\n\ndef workflow(job):\n    """The ComfyUI API graph for one text-to-image job (Turbo LoRA for "fast")."""\n    graph = {\n        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": os.path.basename(IMAGE_FILES[0][1]), "weight_dtype": "default"}},\n        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": os.path.basename(IMAGE_FILES[1][1]), "type": "qwen_image", "device": "default"}},\n        "3": {"class_type": "VAELoader", "inputs": {"vae_name": os.path.basename(IMAGE_FILES[2][1])}},\n        "4": {"class_type": "TextEncodeQwenImage21", "inputs": {"clip": ["2", 0], "prompt": job["prompt"], "negative_prompt": job.get("negative") or "", "resolution": 1024}},\n        "5": {"class_type": "EmptyLatentImage", "inputs": {"width": job["width"], "height": job["height"], "batch_size": 1}},\n        "6": {"class_type": "KSampler", "inputs": {\n            "model": ["9", 0] if job["speed"] == "fast" else ["1", 0],\n            "positive": ["4", 0], "negative": ["4", 1], "latent_image": ["5", 0],\n            "seed": job["seed"], "steps": job["steps"], "cfg": job["cfg"],\n            "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,\n        }},\n        "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}},\n        "8": {"class_type": "SaveImage", "inputs": {"images": ["7", 0], "filename_prefix": job["id"]}},\n    }\n    if job["speed"] == "fast":\n        graph["9"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": os.path.basename(IMAGE_FILES[3][1]), "strength_model": 1.0}}\n    return graph\n\n\ndef sampling_step(mark):\n    """The latest sampling step ComfyUI\'s progress bar printed since `mark` (a log offset), or -1."""\n    try:\n        with open(COMFY_LOG, "rb") as f:\n            f.seek(mark)\n            found = re.findall(r"(\\d+)/(\\d+) \\[", f.read().decode("utf-8", "replace"))\n    except OSError:\n        return -1\n    return int(found[-1][0]) if found else -1\n\n\ndef report_mode(target, mode, state, detail=None, percent=None):\n    try:\n        relay("POST", "/_relay/mode", {"target": target, "mode": mode, "state": state, "detail": detail, "percent": percent})\n    except Exception as err:\n        log(f"mode report failed: {err}")\n\n\ndef unload_chat_model():\n    http("POST", OLLAMA + "/api/generate", {"model": MODEL, "keep_alive": 0}, timeout=120)\n    for _ in range(60):\n        if not http("GET", OLLAMA + "/api/ps").get("models"):\n            return\n        time.sleep(1)\n\n\ndef switch_to_pictures(pictures, url, context):\n    """Hands the GPUs to ComfyUI; on failure gives them back to the chat model. True when it worked."""\n    try:\n        report_mode(url, "images", "switching", "Unloading the chat model", 5)\n        pictures.prepare()\n        while not pictures.ready.wait(5):\n            report_mode(url, "images", "switching", pictures.status, max(10, min(85, pictures.percent)))\n        if pictures.error:\n            raise RuntimeError(pictures.error)\n        unload_chat_model()\n        report_mode(url, "images", "switching", "Starting the picture model", 90)\n        pictures.start()\n        report_mode(url, "images", "ready", "Ready for pictures", 100)\n        log("pictures have the GPUs")\n        return True\n    except Exception as err:\n        message = public(f"{type(err).__name__}: {err}" if not isinstance(err, RuntimeError) else err)\n        log(f"switching to pictures failed: {message}")\n        pictures.stop()\n        report_mode(url, "images", "failed", message)\n        load(context)\n        return False\n\n\ndef switch_to_chat(pictures, url, context):\n    report_mode(url, "chat", "switching", "Freeing the GPUs", 20)\n    pictures.stop()\n    report_mode(url, "chat", "switching", "Loading the chat model", 50)\n    load(context)\n    report_mode(url, "chat", "ready", "Ready to chat", 100)\n    log("the chat model has the GPUs again")\n\n\ndef run_job(job):\n    """Makes one picture, reporting each step; failures are reported to the page."""\n    def keep_going(step):\n        try:\n            return not relay("POST", "/_relay/image-progress", {"id": job["id"], "step": step}).get("cancel")\n        except Exception:\n            return True\n\n    log(f"picture {job[\'id\']}: {job[\'width\']}x{job[\'height\']}, {job[\'steps\']} steps")\n    started = time.time()\n    try:\n        PICTURES.make(job, keep_going)\n        log(f"picture {job[\'id\']} done in {time.time() - started:.0f} s")\n    except Exception as err:\n        message = public(err)\n        log(f"picture {job[\'id\']} failed: {message} | {tail(COMFY_LOG, 3)}")\n        try:\n            upload(f"/_relay/image-result?id={job[\'id\']}&kind=error", json.dumps({"error": message}).encode(), "application/json")\n        except Exception:\n            pass\n\n\n# ----------------------------------------------------------------------------------- serving\n\n\ndef start_tunnel():\n    subprocess.run(["pkill", "-x", "cloudflared"], check=False)\n    time.sleep(1)\n    subprocess.Popen(\n        [CLOUDFLARED_BIN, "tunnel", "--no-autoupdate", "--url", STREAM_PROXY, "--http-host-header", "localhost:11435"],\n        stdout=open("/tmp/cloudflared.log", "w"),\n        stderr=subprocess.STDOUT,\n    )\n    url = None\n    for _ in range(60):\n        time.sleep(2)\n        found = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", tail("/tmp/cloudflared.log", 200))\n        if found:\n            url = found.group(0)\n            break\n    if not url:\n        raise RuntimeError("The Cloudflare tunnel didn\'t start: " + tail("/tmp/cloudflared.log"))\n    for _ in range(45):  # the new hostname takes a few seconds to resolve everywhere\n        if reachable(url):\n            return url\n        time.sleep(2)\n    return url\n\n\ndef reachable(url):\n    try:\n        http("GET", url + "/api/version", timeout=15)\n        return True\n    except Exception:\n        return False\n\n\ndef gpu_busy():\n    try:\n        out = subprocess.run(\n            ["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],\n            capture_output=True, text=True, timeout=20,\n        ).stdout\n        return any(int(v) > BUSY_GPU_PERCENT for v in out.split() if v.strip().isdigit())\n    except Exception:\n        return False\n\n\ndef requests_served():\n    """How many chat and generate requests Ollama has finished (its access log lines)."""\n    try:\n        with open(OLLAMA_LOG, encoding="utf-8", errors="replace") as f:\n            return len(re.findall(r\'POST\\s+"/api/(?:chat|generate)"\', f.read()))\n    except OSError:\n        return 0\n\n\ndef serve(context, vision, quant, url):\n    mode = "chat"\n\n    def register(url):\n        relay("POST", "/_relay/register", {"target": url, "host": RUNTIME_HOST, "num_ctx": context, "num_batch": BATCH_SIZE, **({"profile_id": PROFILE_ID} if PROFILE_ID else {}), **({"job_max_leg_seconds": 300, "job_control": True} if AGENT_ONLY else {}), "inference_transport": "jobs-v1", "vision": vision, "quant": quant, "control": True, "mode": mode})\n        log(f"READY at {url}: {context // 1024}K context, pictures {\'on\' if vision else \'off\'}, {quant}")\n\n    report("connecting", "Connecting to this page")\n    register(url)\n\n    if not AGENT_ONLY:\n        PICTURES.prepare()  # in the background, so the first switch to pictures is quick\n    served = requests_served()\n    last_used = time.time()\n    last_tunnel_check = time.time()\n    tunnel_failures = 0\n    reason = "the 12-hour limit is near"\n    while time.time() - STARTED < MAX_RUNTIME_S:\n        time.sleep(IMAGE_POLL_S if mode == "images" else CHAT_POLL_S)\n        now_served = requests_served()\n        if now_served != served or gpu_busy():\n            served = now_served\n            last_used = time.time()\n\n        # Personal sessions retain a local quota backstop even if the relay is unreachable\n        # or its temporary credentials expire. Owner notebook idle behavior stays unchanged.\n        if AGENT_ONLY and time.time() - last_used >= 120 * 60:\n            reason = "personal coding session had no local client or model activity for 2 hours"\n            break\n\n        try:\n            idle_s = int(time.time() - last_used)\n            jobs = globals().get("PERSONAL_JOBS") if AGENT_ONLY else None\n            active = "".join("&active_generation=" + owner for owner in jobs.active_generations()) if jobs else ""\n            command = relay("GET", f"/_relay/command?target={urllib.parse.quote(url, safe=\'\')}&idle={idle_s}{active}")\n            if jobs and isinstance(command, dict):\n                jobs.cancel_generations(command.get("stop_generations", []))\n        except Exception as err:\n            log(f"couldn\'t reach the relay: {err}")\n            command = None\n        if command:\n            if command.get("shutdown"):\n                reason = "another server took over" if command.get("reason") == "replaced" else "the website asked it to stop"\n                break\n            if AGENT_ONLY and command.get("client_active") is True:\n                client_idle = command.get("client_idle_seconds")\n                if isinstance(client_idle, (int, float)) and not isinstance(client_idle, bool) and 0 <= client_idle <= 120:\n                    last_used = max(last_used, time.time() - client_idle)\n            if not command.get("registered"):\n                register(url)\n            wanted = command.get("mode") or "chat"\n            if wanted == "images" and mode == "chat":\n                if switch_to_pictures(PICTURES, url, context):\n                    mode = "images"\n                last_used = time.time()\n            elif wanted == "chat" and mode == "images":\n                switch_to_chat(PICTURES, url, context)\n                mode = "chat"\n                last_used = time.time()\n            elif command.get("job") and mode == "images":\n                run_job(command["job"])\n                last_used = time.time()\n            idle = command.get("idle_minutes") or 0\n            if idle and time.time() - last_used > idle * 60:\n                reason = f"nobody used it for {idle} minutes"\n                break\n\n        if time.time() - last_tunnel_check < TUNNEL_CHECK_S:\n            continue\n        last_tunnel_check = time.time()\n        if reachable(url):\n            tunnel_failures = 0\n        else:\n            tunnel_failures += 1\n            if tunnel_failures >= 2:\n                log("The tunnel dropped; opening a new one")\n                url = start_tunnel()\n                register(url)\n                tunnel_failures = 0\n\n    PICTURES.stop()\n    log(f"Shutting down: {reason}.")\n    try:\n        relay("POST", "/_relay/unregister", {"target": url})\n    except Exception as err:\n        log(f"sign-off failed: {err}")\n\n\ndef main():\n    """\n    Starts everything that doesn\'t wait on something else at once: the download with the install,\n    the tunnel with the model\'s load. Then serves.\n    """\n    global OPLOG\n    OPLOG = NotebookLogReporter(RELAY, RUNTIME_HOST, headers={RELAY_HEADER: LAUNCH_TOKEN}, secrets=(LAUNCH_TOKEN,))\n    started = time.time()\n    log("Heretic Qwen server starting on " + ("Colab" if RUNTIME_HOST == "colab" else "Kaggle"))\n    if AGENT_ONLY:\n        # The helper may have been closed while this notebook was still queued. Stop before\n        # installing/downloading/loading anything if its isolated session requested shutdown.\n        try:\n            if relay("GET", "/_relay/command").get("shutdown"):\n                log("Personal coding session was cancelled before startup; stopping now")\n                return\n        except Exception:\n            raise RuntimeError("Could not confirm the personal coding session. Stopping without loading a model.") from None\n    models, free_bytes = models_dir()\n    global PICTURES, PROFILE_ID, KNOWN_VISION\n    cache = discover_cache(repo=REPO, quant=QUANT, model_name=MODEL, cache_id=CACHE_ID or None)\n    cached_model = False\n    if cache:\n        try:\n            stage_cached_model(cache, models)\n            cached_model = cache[\'manifest\'].get(\'schema\') != 2\n            log(f"Text model cache hit: {cache[\'manifest\'][\'cache_id\']}; no text-model downloads")\n        except Exception as err:\n            log(f"Text model cache unavailable ({public(err)}); downloading normally")\n    else:\n        log("No matching model cache mounted; downloading normally")\n    if not cached_model:\n        PROFILE_ID = ""\n        KNOWN_VISION = None\n    PICTURES = Pictures(os.path.dirname(models), cache)\n    fetched = None if cached_model else Background(lambda: fetch_model(models, free_bytes))\n    install(cache, os.path.join(os.path.dirname(models), "heretic-runtime"))\n    start_ollama(models)\n    log(f"runtime ready in {time.time() - started:.0f} s")\n    if cached_model:\n        try:\n            http("POST", OLLAMA + "/api/show", {"model": MODEL}, timeout=120)\n            quant = QUANT\n        except Exception as err:\n            log(f"Ollama could not read the cached model ({public(err)}); downloading normally")\n            PROFILE_ID = ""\n            KNOWN_VISION = None\n            # Remove only this cache\'s links; fallback must never write through the read-only mount.\n            for entry in cache["manifest"]["model"]["files"]:\n                path = os.path.join(models, "blobs", "sha256-" + entry["sha256"])\n                if os.path.islink(path) and os.path.realpath(path) == _cache_inside(cache["root"], entry["path"]):\n                    os.unlink(path)\n            fetched = Background(lambda: fetch_model(models, shutil.disk_usage(models).free))\n            quant = get_model(models, fetched)\n    else:\n        quant = get_model(models, fetched)\n    log(f"model ready to load at {time.time() - started:.0f} s")\n    # Job envelopes add a small identity/path wrapper around the original 4 MiB payload.\n    proxy = start_stream_proxy(OLLAMA, port=11435, max_request_bytes=4 * 1024 * 1024 + 1024)\n    try:\n        global PERSONAL_JOBS\n        PERSONAL_JOBS = install_async_job_routes(proxy, max_job_seconds=7200, max_stream_seconds=300, control_token=LAUNCH_TOKEN) if AGENT_ONLY else None\n        if not AGENT_ONLY:\n            install_async_job_routes(proxy)\n        tunnel = Background(start_tunnel)\n        report("loading", "Loading the model onto the GPUs")\n        context = load_best()\n        vision = False if AGENT_ONLY else (True if KNOWN_VISION else sees_pictures(context))\n        url = tunnel.result()\n        log(f"up in {time.time() - started:.0f} s")\n        serve(context, vision, quant, url)\n    finally:\n        proxy.shutdown()\n        proxy.server_close()\n\n\nif __name__ == "__main__":\n    final_status = "COMPLETE"\n    try:\n        main()\n    except Exception as err:\n        final_status = "ERROR"\n        report("failed", public(f"{type(err).__name__}: {err}" if not isinstance(err, RuntimeError) else err))\n        raise\n    finally:\n        if OPLOG is not None:\n            OPLOG.finish(final_status, wait=6)\n    sys.exit(0)\n'

"""Colab adapter for the same checked-in runtime as Kaggle, including Images."""
import getpass
import os
import shutil
import subprocess
import time
import types
import urllib.error
from urllib.parse import urlsplit

RELAY = ""
RELAY_KEY = ""
MODEL_CONTEXT = 32768
MODEL_QUANT = "Q6_K"
MODEL_REPO = "HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive"
MODEL_REVISION = "0a41c68809d375475f954be12ba7c40efa56c2a9"
BATCH_SIZE = 128
config = {"CONTEXT": MODEL_CONTEXT, "MODEL_DIR": "/content/ollama-models", **(globals().get("COLAB_CONFIG") or {})}


def configured_relay():
    value = config.get('RELAY') or RELAY or input('Your deployed relay URL (https://your-worker.example): ').strip()
    parsed = urlsplit(value)
    if parsed.scheme != 'https' or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment or parsed.path not in ('', '/'):
        raise ValueError('Enter your own HTTPS deployment origin, without a path or credentials.')
    return parsed.scheme + '://' + parsed.netloc


def relay_key():
    """Read only this named Colab secret or a hidden prompt."""
    value = ""
    try:
        from google.colab import userdata
        value = userdata.get("QWEN_RELAY_KEY") or ""
    except Exception:
        pass
    if not value:
        value = getpass.getpass("Qwen relay key (hidden; add QWEN_RELAY_KEY in Colab Secrets to remember it): ")
    if not isinstance(value, str) or not 16 <= len(value.strip()) <= 4096:
        raise RuntimeError("Enter the relay key in the hidden input or the QWEN_RELAY_KEY Colab secret.")
    return value.strip()


def gpus():
    try:
        result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.free", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30, check=True)
        return [(name.strip(), int(memory.strip())) for name, memory in (line.rsplit(",", 1) for line in result.stdout.strip().splitlines() if "," in line)]
    except Exception:
        return []


def models_dir():
    selected = config.get("MODEL_DIR")
    if not isinstance(selected, str) or not os.path.isabs(selected):
        raise ValueError("Colab MODEL_DIR must be an absolute directory in this runtime.")
    selected = os.path.abspath(selected)
    os.makedirs(os.path.join(selected, "blobs"), exist_ok=True)
    return selected, shutil.disk_usage(selected).free


def configured_runtime(key):
    if config.get("CONTEXT") not in (8192, 16384, 24576, 32768, 65536):
        raise ValueError("Choose a Colab context of 8192, 16384, 24576, 32768 or 65536 tokens.")
    source = globals().get("COLAB_RUNTIME_SOURCE")
    if not isinstance(source, str) or "RUNTIME_HOST" not in source:
        raise RuntimeError("The shared runtime is missing. Download the current Colab notebook from your website.")
    runtime = {"__name__": "colab_shared_runtime"}
    exec(compile(source, "<qwen-shared-runtime>", "exec"), runtime)
    runtime.update({"RELAY": config.get('RELAY') or RELAY, "LAUNCH_TOKEN": key, "RUNTIME_HOST": "colab", "RELAY_HEADER": "x-relay-key",
                    "STRICT_PROFILE": True, "REPO": MODEL_REPO, "QUANT": MODEL_QUANT, "FALLBACK_QUANT": MODEL_QUANT,
                    "CONTEXTS": [n for n in (65536, 32768, 24576, 16384, 8192) if n <= config["CONTEXT"]], "BATCH_SIZE": BATCH_SIZE,
                    "AGENT_ONLY": False, "CACHE_ID": "", "KNOWN_VISION": None, "models_dir": models_dir,
                    "USER_AGENT": "heretic-qwen-colab/2", "STARTED": time.time()})
    runtime["discover_cache"] = lambda **kwargs: None
    # Bind Python defaults as well as globals: the shared downloader was defined for Kaggle's repo.
    download, signed = runtime["download"], runtime["signed_url"]
    runtime["download"] = lambda files, dest_for, repo=MODEL_REPO, **kwargs: download(files, dest_for, repo=repo, **kwargs)
    def signed_url(name, repo=MODEL_REPO):
        if repo != MODEL_REPO: return signed(name, repo)
        request = urllib.request.Request(f"https://huggingface.co/{repo}/resolve/{MODEL_REVISION}/{name}", headers={"range": "bytes=0-0", "user-agent": runtime["USER_AGENT"]})
        with urllib.request.urlopen(request, timeout=60) as response: return response.geturl()
    runtime["signed_url"] = signed_url
    def model_files(quant):
        if quant != MODEL_QUANT: raise ValueError("Unexpected Colab precision.")
        tree = runtime["http"]("GET", f"https://huggingface.co/api/models/{MODEL_REPO}/tree/{MODEL_REVISION}")
        names = [f"Qwen3.5-9B-Uncensored-HauhauCS-Aggressive-{quant}.gguf", "mmproj-Qwen3.5-9B-Uncensored-HauhauCS-Aggressive-BF16.gguf"]
        files = [next(item for item in tree if item.get("path") == name and item.get("lfs")) for name in names]
        return [(item["path"], item["lfs"]["oid"], item["lfs"]["size"]) for item in files]
    runtime["model_files"] = model_files
    def load_profile():
        for context in runtime["CONTEXTS"]:
            runtime["report"]("loading", f"Loading Qwen 9B Q6_K at {context:,} context tokens")
            try:
                if runtime["load"](context): return context
            except Exception as error: runtime["log"]("GPU fit check: " + runtime["public"](str(error)))
            runtime["log"](f"{context:,} tokens did not fit fully on GPU; trying a smaller context with the same 9B Q6_K model.")
        raise RuntimeError("Qwen 9B Q6_K could not fit entirely on this Colab GPU, even at 8,192 context tokens. Free GPU memory or reconnect a GPU runtime; no CPU fallback was used.")
    runtime["load_best"] = load_profile
    return runtime


def preflight(runtime):
    machine = gpus()
    if not machine:
        raise RuntimeError("No GPU found. Select a GPU runtime in Colab. Qwen 9B requires enough free GPU memory.")
    files = runtime["model_files"](MODEL_QUANT)
    weights = sum(size for _name, _sha, size in files)
    available = sum(memory for _name, memory in machine) * 1024 * 1024
    if available < weights + 1024 * 1024 * 1024:
        raise RuntimeError(f"Qwen 9B Q6_K cannot fit: {available / 1e9:.1f} GB GPU memory is free, while selected model files alone need {weights / 1e9:.1f} GB before context and runtime overhead. Free GPU memory or select a GPU with more memory. The 9B model was not downloaded.")
    runtime["log"]("Colab profile: Qwen 9B Q6_K, up to 32,768 context by default, batch 128, q8_0 KV; chat, vision and Images mode.")
    runtime["log"]("Model-file memory preflight passed. Context is measured on this GPU and reduced only if needed; model precision stays Q6_K.")
    def exact_files(quant):
        if quant != MODEL_QUANT:
            raise ValueError("Unexpected precision.")
        return files
    runtime["model_files"] = exact_files
    return machine


def main():
    global RELAY_KEY, RELAY
    RELAY = configured_relay()
    RELAY_KEY = relay_key()
    runtime = configured_runtime(RELAY_KEY)
    reporter = runtime["NotebookLogReporter"](RELAY, "colab", headers={"x-relay-key": RELAY_KEY}, secrets=(RELAY_KEY,))
    runtime["OPLOG"] = reporter
    runtime["NotebookLogReporter"] = lambda *args, **kwargs: reporter
    registered_target = None
    original_relay = runtime["relay"]
    def tracked_relay(method, route, body=None, **kwargs):
        nonlocal registered_target
        if method == "POST" and route == "/_relay/register" and isinstance(body, dict): body = {**body, "model_size": "9B"}
        result = original_relay(method, route, body, **kwargs) if body is not None else original_relay(method, route, **kwargs)
        if method == "POST" and isinstance(body, dict):
            if route == "/_relay/register": registered_target = body.get("target")
            elif route == "/_relay/unregister" and body.get("target") == registered_target: registered_target = None
        return result
    runtime["relay"] = tracked_relay
    started = []
    original_popen = subprocess.Popen
    def tracked_popen(*args, **kwargs):
        child = original_popen(*args, **kwargs)
        started.append(child)
        return child
    runtime["subprocess"] = types.SimpleNamespace(**{name: getattr(subprocess, name) for name in dir(subprocess) if name != "Popen"}, Popen=tracked_popen)
    final_status = "COMPLETE"
    try:
        runtime["relay"]("GET", "/_relay/command")
        runtime["report"]("installing", "Checking GPU capacity for the Colab Qwen 9B profile")
        preflight(runtime)
        runtime["main"]()
    except KeyboardInterrupt:
        runtime["log"]("Colab was stopped from the notebook")
    except Exception as error:
        final_status = "ERROR"
        if isinstance(error, urllib.error.HTTPError) and error.code in (401, 403):
            message = "The relay rejected the key. Check the QWEN_RELAY_KEY Colab secret."
        else:
            message = runtime["notebook_log_redact"](str(error), (RELAY_KEY,))
        runtime["report"]("failed", runtime["public"](message))
        runtime["log"](message)
    finally:
        if registered_target:
            try: original_relay("POST", "/_relay/unregister", {"target": registered_target})
            except Exception: pass
        pictures = runtime.get("PICTURES")
        if pictures is not None:
            try: pictures.stop()
            except Exception: pass
        for child in reversed(started):
            try:
                if child.poll() is None:
                    child.terminate()
                    try: child.wait(timeout=10)
                    except subprocess.TimeoutExpired: child.kill()
            except Exception:
                pass
        reporter.finish(final_status, wait=6)


if __name__ == "__main__":
    main()
